---

## Corr_Summary Segmentation Fix - Documentation

### Root Cause

The `build_all_summary_corrs_segmented()` function was attempting to discover and filter segment values from `df_summary_clean` using operations like:
- `df_summary_clean.select(seg_dim).distinct().collect()`
- `df_summary_clean.filter(F.col(seg_dim) == seg_val)`

However, `df_summary_clean` was built by selecting only `summary_keep_cols`, which originally contained:
- Time column (`SUMMARY_TIME_COL`)
- Summary metric columns (`SUMMARY_ALL_FEATURE_COLS`)

The segment dimension columns from `SEGMENT_DIMS` were **not included** in `summary_keep_cols`, causing the segmented correlation logic to fail when trying to access columns that didn't exist in the DataFrame.

### What Was Changed

**Location:** Cell #9 (SUMMARY CORRELATIONS: P50 / P95 / P99)

**Changes made:**

1. **Segment Column Discovery** (after `summary_keep_cols` construction):
   - Added code to discover which `SEGMENT_DIMS` columns exist in `df_summary`
   - Appended these columns to `summary_keep_cols` before creating `df_summary_clean`
   - Added informative print statements showing which segment columns were added

2. **Pre-Build Validation** (after `df_summary_clean` creation):
   - Added validation showing which segment columns are available in `df_summary_clean`
   - Added bounded sample collection showing example values for first 2 segment dimensions
   - Helps verify the fix worked before expensive correlation computation

3. **Post-Build Validation** (after `corr_summary_pdf` creation):
   - Added comprehensive validation of the output structure
   - Validates `segment_dim` and `segment_value` columns exist
   - Shows row counts for baseline (all/all) and each segmented dimension
   - Displays sample segment values to prove segmentation worked

### Fix Method

**Expanded `summary_keep_cols` to include segment columns.**

This approach was chosen because:
- Minimal change to existing architecture
- Preserves the single `df_summary_clean` DataFrame contract
- No need for separate DataFrames or complex branching logic
- Maintains backward compatibility (empty `SEGMENT_DIMS` = no extra columns)

### What Downstream Notebooks Can Now Assume

**The `dcgm_eda_corr_summary` table now reliably contains:**

1. **Baseline rows** with `segment_dim = 'all'` and `segment_value = 'all'`
   - Global correlations across all data (preserves original behavior)

2. **Segmented rows** for each dimension in `SEGMENT_DIMS`:
   - `segment_dim = 'gen'` with values for each GPU generation
   - `segment_dim = 'region_rollup'` with values 'EU' and 'NAM'
   - `segment_dim = 'customer_segment'` with customer segment values
   - `segment_dim = 'product_segment'` with product segment values
   - `segment_dim = 'is_training'` with training status values

3. **Segmentation is one-dimensional** (not cross-product):
   - Each row belongs to exactly one segment dimension
   - No multi-dimensional combinations (e.g., no gen × region × customer)

4. **All original correlation columns preserved**:
   - `metric_x`, `metric_y`
   - `correlation_value`, `method`
   - `summary_percentile`
   - Plus the new `segment_dim` and `segment_value`

**Downstream filtering examples:**

```python
# Get baseline global correlations
baseline = df_corr.filter((F.col("segment_dim") == "all") & (F.col("segment_value") == "all"))

# Get all gen-segmented correlations
gen_corrs = df_corr.filter(F.col("segment_dim") == "gen")

# Get correlations for a specific region
eu_corrs = df_corr.filter((F.col("segment_dim") == "region_rollup") & (F.col("segment_value") == "EU"))
```

### Testing Status

✅ **Validation code added** to verify the fix works at runtime  
✅ **No changes to segmentation logic** - only fixed the input DataFrame contract  
✅ **Backward compatible** - empty `SEGMENT_DIMS` still works (produces baseline only)  

---

# GPU Metrics Exploratory Data Analysis - Sparkcaster Edition

This notebook performs exploratory analysis on GPU utilization metrics using Sparkcaster + Nessie, with a cleaned execution path for summary correlations, raw validation, density binning, trend summaries, and persistence validation.

---

## Notebook structure

### 1. Setup and configuration
- Cell 1: Package installation and version-aware dependency checks
- Cell 2: CAIOS credentials setup
- Cell 3: Sparkcaster + Nessie initialization
- Cell 4: Runtime configuration

### 2. Data loading and schema resolution
- Cell 5: Load raw and summary tables
- Cell 6: Source validation and summary-feature resolution
- Cell 7: Create sampled raw data for validation workflows

### 3. Correlation analysis
- Cell 8: Summary correlations across percentile tiers
- Cell 9: Raw-level correlation validation on sampled data
- Cell 10: Correlations over time
- Cell 11: Persist correlation outputs

### 4. Density analysis
- Cell 12: Fixed-bin density helper functions
- Cell 13: Optional HoloViews rendering helpers
- Cell 14: Static density computation
- Cell 15: Density over time computation
- Cell 16: Persist density outputs

### 5. Trend analysis and validation
- Cell 17: Trend summary from summary table
- Cell 18: Persist trend outputs
- Cell 19: Persistence validation
- Cell 20: Comprehensive read-back validation
- Cell 21: Spark cluster diagnostics
- Cell 22: Final cleanup

---

## Notes
- Stale duplicate cells from earlier refactors were removed.
- Correlation outputs now consistently use `summary_feature_group`.
- Density outputs are explicitly persisted before validation.
- The notebook is intended to run top-to-bottom in the cleaned order above.


In [1]:
# ========================================
# PACKAGE INSTALLATION
# ========================================
# Install only packages needed for Sparkcaster + Nessie + visualization
# Removed: StarRocks-specific packages (starrocks, sqlalchemy, pymysql, oauth2client)

import importlib
import importlib.metadata as importlib_metadata
import subprocess
import sys

def is_module_available(module_name: str) -> bool:
    """Return True if the importable module exists."""
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False

def install_if_missing(pip_name: str, import_name: str = None):
    """Install pip_name if missing, or upgrade when an exact pin is specified and mismatched."""
    import_name = import_name or pip_name
    dist_name = pip_name.split("==", 1)[0]
    required_version = pip_name.split("==", 1)[1] if "==" in pip_name else None

    if not is_module_available(import_name):
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", pip_name])
        return

    if required_version:
        try:
            installed_version = importlib_metadata.version(dist_name)
        except importlib_metadata.PackageNotFoundError:
            installed_version = None
        if installed_version != required_version:
            print(f"Upgrading {dist_name} from {installed_version or 'unknown'} to {required_version} ...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "--upgrade", pip_name])
            return

    print(f"✓ {pip_name} already installed")

print("="*60)
print("INSTALLING DEPENDENCIES")
print("="*60)

# Core dependencies
install_if_missing("keyring", "keyring")
install_if_missing("keyrings.cryptfile", "keyrings.cryptfile")
install_if_missing("pyarrow", "pyarrow")
install_if_missing("fsspec", "fsspec")
install_if_missing("s3fs", "s3fs")

# Analysis and ML
install_if_missing("numpy", "numpy")
install_if_missing("pandas", "pandas")
install_if_missing("scikit-learn", "sklearn")
install_if_missing("statsmodels", "statsmodels")

# Visualization - pinned versions for Python 3.11 compatibility
install_if_missing("matplotlib", "matplotlib")
install_if_missing("bokeh==3.6.2", "bokeh")
install_if_missing("jupyter_bokeh", "jupyter_bokeh")
install_if_missing("panel==1.5.2", "panel")
install_if_missing("holoviews==1.19.0", "holoviews")
install_if_missing("hvplot==0.10.0", "hvplot")
install_if_missing("datashader==0.16.3", "datashader")

print("\n" + "="*60)
print("✓ ALL DEPENDENCIES INSTALLED")
print("="*60)


INSTALLING DEPENDENCIES
✓ keyring already installed
✓ keyrings.cryptfile already installed
✓ pyarrow already installed
✓ fsspec already installed
✓ s3fs already installed
✓ numpy already installed
✓ pandas already installed
✓ scikit-learn already installed
Installing statsmodels ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 41.2 MB/s  0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [statsmodels] [statsmodels]
Installing matplotlib ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 46.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 123.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 243.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━ 4/5 [matplotlib]

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [matplotlib]5 [matplotlib]
Installing bokeh==3.6.2 ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 33.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [bokeh]32m1/2 [bokeh]
Installing jupyter_bokeh ...


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 9.7 MB/s  0:00:00m eta 0:00:01
Installing panel==1.5.2 ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 39.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 5/6 [panel]

...

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━  9/11 [dask]y]ckle]

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [datashader]1 [datashader]

✓ ALL DEPENDENCIES INSTALLED


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
# ========================================
# CAIOS CREDENTIALS SETUP
# ========================================
# Only CAIOS credentials needed for Nessie access via Sparkcaster
# StarRocks credentials removed - no longer needed

import keyring
import os
from getpass import getpass
from keyrings.cryptfile.cryptfile import CryptFileKeyring
from pathlib import Path

print("="*60)
print("CAIOS CREDENTIALS SETUP")
print("="*60)

# Configure encrypted keyring
home_dir = str(Path.home())
os.environ["KEYRING_CRYPTFILE_PATH"] = f"{home_dir}/.local/share/python_keyring/cryptfile_pass.cfg"

kr = CryptFileKeyring()
kr.keyring_key = getpass("Set/enter master password for encrypted keyring: ")
keyring.set_keyring(kr)

# Retrieve or prompt for CAIOS credentials
caios_access_key = keyring.get_password("caios", "access_key")
caios_secret_key = keyring.get_password("caios", "secret_key")

if not caios_access_key:
    caios_access_key = input("Enter CAIOS access key: ")
    keyring.set_password("caios", "access_key", caios_access_key)

if not caios_secret_key:
    caios_secret_key = getpass("Enter CAIOS secret key: ")
    keyring.set_password("caios", "secret_key", caios_secret_key)

print("\n✓ CAIOS credentials configured")
print("="*60)


CAIOS CREDENTIALS SETUP

✓ CAIOS credentials configured


In [3]:
# ========================================
# SPARKCASTER + NESSIE INITIALIZATION
# ========================================
from spark.nessie.client import NessieSparkClient
from pyspark.sql import SparkSession

print("="*60)
print("SPARKCASTER + NESSIE INITIALIZATION")
print("="*60)

# Clean up existing Spark session if present
try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        print("Stopping existing Spark session...")
        existing_spark.stop()
        print("✓ Existing session stopped")
except Exception as e:
    print(f"Note: {e}")

# Initialize NessieSparkClient with DBTCaster mode
ness = NessieSparkClient(
    svc_url="http://kf-proxy.nessie.svc.cluster.local:19120/api/v2",
    nessie_endpoint="http://nessie-prd.cwobject.com",
    caios_access_key=caios_access_key,
    caios_secret_key=caios_secret_key,
    dbtcaster=True,
)

spark = ness.spark
spark.sparkContext.setLogLevel("ERROR")

# Display cluster configuration
print("\n" + "="*60)
print("SPARK CLUSTER CONFIGURATION")
print("="*60)
print(f"Spark Version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(f"App Name: {spark.sparkContext.appName}")
print(f"Application ID: {spark.sparkContext.applicationId}")

# Key configuration parameters
config_params = [
    "spark.driver.memory",
    "spark.driver.cores",
    "spark.executor.instances",
    "spark.executor.cores",
    "spark.executor.memory",
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
]

for param in config_params:
    try:
        value = spark.conf.get(param)
        print(f"{param}: {value}")
    except Exception:
        print(f"{param}: <not set>")

# Executor count
try:
    executor_count = spark.sparkContext._jsc.sc().getExecutorMemoryStatus().size()
    print(f"\n✓ Active executors: {executor_count}")
except Exception as e:
    print(f"\nCould not get executor count: {e}")

print("="*60)


SPARKCASTER + NESSIE INITIALIZATION

SPARK CLUSTER CONFIGURATION
Spark Version: 4.1.2
Master: k8s://https://10.16.0.1:443
App Name: sparkcaster-jbok-4df513e2-3d70-43e2-b455-0630944c9842
Application ID: spark-6b6bfd270dad4fc6b7a93ed2fc9d0718
spark.driver.memory: 16g
spark.driver.cores: 4
spark.executor.instances: 10
spark.executor.cores: 4
spark.executor.memory: 64g
spark.sql.shuffle.partitions: 200
spark.sql.adaptive.enabled: true

✓ Active executors: 10


In [4]:
# ========================================
# RUNTIME CONFIGURATION
# ========================================
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql import types as T

print("="*60)
print("RUNTIME CONFIGURATION")
print("="*60)

RAW_TABLE = "sandbox.sandbox_finance.dcgm_metrics_raw_impute_v2"
SUMMARY_TABLE = "sandbox.sandbox_finance.dcgm_metrics_summary_imputed_v2"

OUTPUT_SCHEMA = "sandbox.sandbox_finance"
OUTPUT_PREFIX = "dcgm_eda"

CORE_RAW_METRIC_COLS = ['tensor_util', 'gpu_util', 'tensor_tflops', 'tensor_tflops_sm','chip_power', 'redfish_power', 'sm_active']
EXTENDED_RAW_METRIC_CANDIDATES = [
    "dram_active", "mem_copy_util", "vram_usage", "sm_clock", "sm_occupancy" , "TFLOPS_per_watt_efficiency"
]
INDEX_METRIC_SUFFIX = "_index"

# Backward-compatible alias for cells that expect RAW_METRIC_COLS.
# Keep it constrained to the core set so full density remains bounded.
RAW_METRIC_COLS = CORE_RAW_METRIC_COLS.copy()
RAW_TIME_COL = "datestamp"

SUMMARY_TIME_CANDIDATES = ["day", "date", "datestamp", "timestamp", "ts"]
SUMMARY_PERCENTILES_PRIORITY = ["p50", "p95", "p99"]
SUMMARY_TIME_COL = None
SUMMARY_FEATURE_GROUP = None
SUMMARY_METRIC_COLS = []
SUMMARY_FEATURE_LABEL = None
SUMMARY_SOURCE_MODE = None
SUMMARY_FEATURE_TO_BASE_METRIC = {}
SUMMARY_BASE_METRIC_CANDIDATES = []

AVAILABLE_EXTENDED_RAW_METRIC_COLS = []
AVAILABLE_INDEX_RAW_METRIC_COLS = []
SAMPLE_RAW_METRIC_COLS = []
RAW_VALIDATION_METRIC_COLS = []
DENSITY_METRIC_COLS = CORE_RAW_METRIC_COLS.copy()

RAW_SAMPLE_FRACTION = 0.0005
SAMPLE_SEED = 42
SCATTER_RENDER_LIMIT = 10000

NUM_BINS = 64
PERCENTILE_RANGE = (0.01, 0.99)
TIME_BUCKET_GRAIN = "day"
CORR_METHODS = ["pearson", "spearman"]


# ============================================
# SEGMENT DIMENSIONS CONFIGURATION
# ============================================
# Supported segment dimensions for one-at-a-time analysis
SEGMENT_DIMS = ["gen", "region_rollup", "customer_segment", "product_segment", "is_training"]

RUN_ID = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.utcnow()

print("\nSource Tables:")
print(f"  Raw: {RAW_TABLE}")
print(f"  Summary: {SUMMARY_TABLE}")
print("\nOutput Configuration:")
print(f"  Schema: {OUTPUT_SCHEMA}")
print(f"  Prefix: {OUTPUT_PREFIX}")
print(f"  Run ID: {RUN_ID}")
print("\nMetric Tiers:")
print(f"  Core raw metrics (density default): {CORE_RAW_METRIC_COLS}")
print(f"  Extended raw metric candidates: {EXTENDED_RAW_METRIC_CANDIDATES}")
print(f"  Auto-detected index suffix: {INDEX_METRIC_SUFFIX}")
print(f"Raw Time Column: {RAW_TIME_COL}")
print(f"Summary Time Candidates: {SUMMARY_TIME_CANDIDATES}")
print(f"Summary Percentile Priority: {SUMMARY_PERCENTILES_PRIORITY}")
print("\nSampling:")
print(f"  Raw sample fraction: {RAW_SAMPLE_FRACTION:.4%}")
print(f"  Scatter render limit: {SCATTER_RENDER_LIMIT:,}")
print("\nBinning:")
print(f"  Bins: {NUM_BINS}")
print(f"  Percentile range: {PERCENTILE_RANGE}")
print(f"\nTime Bucketing: {TIME_BUCKET_GRAIN}")
print(f"\nSegmentation:")
print(f"  Segment dimensions: {SEGMENT_DIMS}")
print("="*60)

# ========================================
# TABLE SCHEMA INSPECTION
# ========================================
print("\n" + "="*60)
print("TABLE SCHEMAS")
print("="*60)

print(f"\n{'='*60}")
print(f"RAW_TABLE Schema: {RAW_TABLE}")
print(f"{'='*60}")
spark.table(RAW_TABLE).printSchema()

print(f"\n{'='*60}")
print(f"SUMMARY_TABLE Schema: {SUMMARY_TABLE}")
print(f"{'='*60}")
spark.table(SUMMARY_TABLE).printSchema()

print("="*60)


RUNTIME CONFIGURATION

Source Tables:
  Raw: sandbox.sandbox_finance.dcgm_metrics_raw_impute_v2
  Summary: sandbox.sandbox_finance.dcgm_metrics_summary_imputed_v2

Output Configuration:
  Schema: sandbox.sandbox_finance
  Prefix: dcgm_eda
  Run ID: 20260624_201713

Metric Tiers:
  Core raw metrics (density default): ['tensor_util', 'gpu_util', 'tensor_tflops', 'tensor_tflops_sm', 'chip_power', 'redfish_power', 'sm_active']
  Extended raw metric candidates: ['dram_active', 'mem_copy_util', 'vram_usage', 'sm_clock', 'sm_occupancy', 'TFLOPS_per_watt_efficiency']
  Auto-detected index suffix: _index
Raw Time Column: datestamp
Summary Time Candidates: ['day', 'date', 'datestamp', 'timestamp', 'ts']
Summary Percentile Priority: ['p50', 'p95', 'p99']

Sampling:
  Raw sample fraction: 0.0500%
  Scatter render limit: 10,000

Binning:
  Bins: 64
  Percentile range: (0.01, 0.99)

Time Bucketing: day

Segmentation:
  Segment dimensions: ['gen', 'region_rollup', 'customer_segment', 'product_segment

In [5]:
# ========================================
# DATA LOADING FROM NESSIE
# ========================================
import numpy as np
import pandas as pd

print("="*60)
print("LOADING DATA FROM NESSIE")
print("="*60)

# Load raw table (do NOT cache yet - too large)
print(f"\nLoading raw table: {RAW_TABLE}")
df_raw = ness.sql(f"SELECT * FROM {RAW_TABLE}")
print("✓ Raw table loaded (not cached - will sample for analysis)")

# Load summary table (smaller, can cache if reused)
print(f"\nLoading summary table: {SUMMARY_TABLE}")
df_summary = ness.sql(f"SELECT * FROM {SUMMARY_TABLE}")
print("✓ Summary table loaded")

# Display schemas
print("\n" + "="*60)
print("RAW TABLE SCHEMA")
print("="*60)
df_raw.printSchema()

print("\n" + "="*60)
print("SUMMARY TABLE SCHEMA")
print("="*60)
df_summary.printSchema()

# Display sample rows
print("\n" + "="*60)
print("RAW TABLE SAMPLE (5 rows)")
print("="*60)
# df_raw.show(5, truncate=False)

print("\n" + "="*60)
print("SUMMARY TABLE SAMPLE (5 rows)")
print("="*60)
# df_summary.show(5, truncate=False)

print("\n✓ Data loading complete")
print("="*60)


LOADING DATA FROM NESSIE

Loading raw table: sandbox.sandbox_finance.dcgm_metrics_raw_impute_v2
✓ Raw table loaded (not cached - will sample for analysis)

Loading summary table: sandbox.sandbox_finance.dcgm_metrics_summary_imputed_v2
✓ Summary table loaded

RAW TABLE SCHEMA
root
 |-- datestamp: timestamp (nullable = true)
 |-- node: string (nullable = true)
 |-- gpu_util: double (nullable = true)
 |-- tensor_util: double (nullable = true)
 |-- chip_power: double (nullable = true)
 |-- dram_active: double (nullable = true)
 |-- mem_copy_util: double (nullable = true)
 |-- vram_usage: double (nullable = true)
 |-- sm_active: double (nullable = true)
 |-- sm_clock: double (nullable = true)
 |-- sm_occupancy: double (nullable = true)
 |-- region: string (nullable = true)
 |-- zone: string (nullable = true)
 |-- cluster: string (nullable = true)
 |-- cluster_org: string (nullable = true)
 |-- cw_sku: string (nullable = true)
 |-- model: string (nullable = true)
 |-- serial: string (nullabl

In [6]:
# ========================================
# SOURCE DATA VALIDATION + SCHEMA RESOLUTION
# ========================================

def detect_summary_time_col(summary_cols, candidates):
    for candidate in candidates:
        if candidate in summary_cols:
            return candidate
    return None


def dedupe_preserve(seq):
    seen = set()
    out = []
    for item in seq:
        if item not in seen:
            seen.add(item)
            out.append(item)
    return out


def resolve_summary_feature_groups(summary_cols, base_metric_candidates, percentiles_priority):
    groups = {}
    base_map = {}
    for pct in percentiles_priority:
        cols = []
        for base_metric in base_metric_candidates:
            candidate = f"{base_metric}_{pct}"
            if candidate in summary_cols:
                cols.append(candidate)
                base_map[candidate] = base_metric
        if cols:
            groups[pct] = cols
    fallback_cols = [metric for metric in base_metric_candidates if metric in summary_cols]
    if fallback_cols:
        groups["raw_fallback"] = fallback_cols
        for col in fallback_cols:
            base_map[col] = col
    return groups, base_map


def normalize_metric_name(summary_metric: str):
    for pct in SUMMARY_PERCENTILES_PRIORITY:
        suffix = f"_{pct}"
        if summary_metric.endswith(suffix):
            return summary_metric[:-len(suffix)]
    return summary_metric


def build_summary_corr_pairs_for_tier(df, metric_cols, method='pearson'):
    import pandas as pd
    pdf = df.select(*metric_cols).toPandas()
    pdf = pdf.apply(pd.to_numeric, errors='coerce')
    pdf = pdf.dropna(axis=1, how='all')
    if pdf.shape[1] < 2:
        return pd.DataFrame(columns=['metric_x', 'metric_y', 'correlation_value', 'method'])
    corr = pdf.corr(method=method)
    rows = []
    cols = list(corr.columns)
    for i, x in enumerate(cols):
        for j, y in enumerate(cols):
            if i >= j:
                continue
            rows.append({
                'metric_x': x,
                'metric_y': y,
                'correlation_value': float(corr.loc[x, y]),
                'method': method,
            })
    return pd.DataFrame(rows)


def build_all_summary_corrs(df_summary_clean, summary_groups):
    import pandas as pd
    frames = []
    for pct, cols in summary_groups.items():
        if len(cols) < 2:
            continue
        for method in CORR_METHODS:
            part = build_summary_corr_pairs_for_tier(df_summary_clean, cols, method=method)
            if part.empty:
                continue
            part['summary_percentile'] = pct
            part['base_metric_x'] = part['metric_x'].map(normalize_metric_name)
            part['base_metric_y'] = part['metric_y'].map(normalize_metric_name)
            frames.append(part)
    if not frames:
        return pd.DataFrame(columns=['metric_x', 'metric_y', 'base_metric_x', 'base_metric_y', 'correlation_value', 'method', 'summary_percentile'])
    return pd.concat(frames, ignore_index=True)


def build_auto_validation_pairs(corr_pdf, top_n=5):
    if corr_pdf.empty:
        return []
    tmp = corr_pdf.copy()
    tmp['abs_corr'] = tmp['correlation_value'].abs()
    tmp = tmp.sort_values(['summary_percentile', 'method', 'abs_corr'], ascending=[True, True, False])
    tmp = tmp.groupby(['summary_percentile', 'method'], as_index=False).head(top_n)
    out = []
    for _, r in tmp.iterrows():
        out.append({
            'summary_percentile': r['summary_percentile'],
            'method': r['method'],
            'summary_metric_x': r['metric_x'],
            'summary_metric_y': r['metric_y'],
            'base_metric_x': r['base_metric_x'],
            'base_metric_y': r['base_metric_y'],
            'summary_correlation': r['correlation_value'],
        })
    return out

print("="*60)
print("SOURCE DATA VALIDATION")
print("="*60)

raw_cols = set(df_raw.columns)
summary_cols = set(df_summary.columns)

# ========================================
# VALIDATE SEGMENT DIMENSIONS
# ========================================
if SEGMENT_DIMS:
    missing_dims = [dim for dim in SEGMENT_DIMS if dim not in raw_cols]
    if missing_dims:
        print(f"WARNING: Segment dimensions not found in raw data: {missing_dims}")
        SEGMENT_DIMS = [dim for dim in SEGMENT_DIMS if dim in raw_cols]

    summary_missing_dims = [dim for dim in SEGMENT_DIMS if dim not in summary_cols]
    if summary_missing_dims:
        print(f"WARNING: Segment dimensions not found in summary data: {summary_missing_dims}")
        SEGMENT_DIMS = [dim for dim in SEGMENT_DIMS if dim in summary_cols]

    if SEGMENT_DIMS:
        print(f"Segment dimensions validated: {SEGMENT_DIMS}")
    else:
        print("No valid segment dimensions found - running global analysis only")
else:
    print("No segment dimensions configured - running global analysis only")


print("\n[1/6] Large-table row counts intentionally skipped (scan-safe)")
print("[2/6] Resolving raw metric tiers...")
missing_core = [col for col in CORE_RAW_METRIC_COLS if col not in raw_cols]
if missing_core:
    print(f"✗ WARNING: Core raw metrics missing from raw table: {missing_core}")
else:
    print(f"✓ All core raw metrics present: {CORE_RAW_METRIC_COLS}")

AVAILABLE_EXTENDED_RAW_METRIC_COLS = [col for col in EXTENDED_RAW_METRIC_CANDIDATES if col in raw_cols]
AVAILABLE_INDEX_RAW_METRIC_COLS = sorted([col for col in raw_cols if col.endswith(INDEX_METRIC_SUFFIX)])
SAMPLE_RAW_METRIC_COLS = dedupe_preserve(CORE_RAW_METRIC_COLS + AVAILABLE_EXTENDED_RAW_METRIC_COLS + AVAILABLE_INDEX_RAW_METRIC_COLS)
DENSITY_METRIC_COLS = [col for col in CORE_RAW_METRIC_COLS if col in raw_cols]
RAW_METRIC_COLS = DENSITY_METRIC_COLS.copy()

print(f"✓ Extended raw metrics available: {AVAILABLE_EXTENDED_RAW_METRIC_COLS}")
print(f"✓ Auto-detected index metrics available: {AVAILABLE_INDEX_RAW_METRIC_COLS}")
print(f"✓ Raw sample metrics (sampled/raw exploratory path): {SAMPLE_RAW_METRIC_COLS}")
print(f"✓ Density metrics (bounded expensive path): {DENSITY_METRIC_COLS}")

if RAW_TIME_COL in raw_cols:
    print(f"✓ Raw time column '{RAW_TIME_COL}' found in raw table")
else:
    print(f"✗ WARNING: Raw time column '{RAW_TIME_COL}' not found in raw table")

print("\n[3/6] Resolving summary schema...")
SUMMARY_TIME_COL = detect_summary_time_col(summary_cols, SUMMARY_TIME_CANDIDATES)
if SUMMARY_TIME_COL:
    print(f"✓ Summary time column resolved to '{SUMMARY_TIME_COL}'")
else:
    print(f"✗ WARNING: Could not resolve summary time column from candidates: {SUMMARY_TIME_CANDIDATES}")

SUMMARY_PERCENTILE_GROUPS = {}
SUMMARY_ALL_FEATURE_COLS = []
SUMMARY_TOP_VALIDATION_PAIRS = 5

SUMMARY_BASE_METRIC_CANDIDATES = dedupe_preserve([
    metric for metric in (CORE_RAW_METRIC_COLS + AVAILABLE_EXTENDED_RAW_METRIC_COLS + AVAILABLE_INDEX_RAW_METRIC_COLS)
    if any(f"{metric}_{pct}" in summary_cols for pct in SUMMARY_PERCENTILES_PRIORITY) or metric in summary_cols
])
summary_groups, SUMMARY_FEATURE_TO_BASE_METRIC = resolve_summary_feature_groups(
    summary_cols,
    SUMMARY_BASE_METRIC_CANDIDATES,
    SUMMARY_PERCENTILES_PRIORITY,
)

if summary_groups:
    SUMMARY_PERCENTILE_GROUPS = {
        pct: cols
        for pct, cols in summary_groups.items()
        if pct in SUMMARY_PERCENTILES_PRIORITY and cols
    }
    SUMMARY_ALL_FEATURE_COLS = dedupe_preserve([c for cols in SUMMARY_PERCENTILE_GROUPS.values() for c in cols])
    preferred_group = next((pct for pct in SUMMARY_PERCENTILES_PRIORITY if pct in summary_groups), None)
    if preferred_group is None and "raw_fallback" in summary_groups:
        preferred_group = "raw_fallback"
    SUMMARY_FEATURE_GROUP = preferred_group
    SUMMARY_METRIC_COLS = summary_groups[preferred_group]
    SUMMARY_SOURCE_MODE = "percentile" if preferred_group != "raw_fallback" else "raw_fallback"
    SUMMARY_FEATURE_LABEL = preferred_group
    print(f"✓ Summary feature group resolved to '{SUMMARY_FEATURE_GROUP}'")
    print(f"✓ Summary base metric candidates: {SUMMARY_BASE_METRIC_CANDIDATES}")
    print(f"✓ Summary features: {SUMMARY_METRIC_COLS}")
else:
    SUMMARY_FEATURE_GROUP = None
    SUMMARY_METRIC_COLS = []
    SUMMARY_SOURCE_MODE = None
    SUMMARY_FEATURE_LABEL = None
    print("✗ WARNING: Could not resolve summary metric columns from summary schema")

print("\n[4/6] Resolving raw validation overlap...")
RAW_VALIDATION_METRIC_COLS = dedupe_preserve([
    SUMMARY_FEATURE_TO_BASE_METRIC.get(col, col)
    for col in SUMMARY_ALL_FEATURE_COLS
    if SUMMARY_FEATURE_TO_BASE_METRIC.get(col, col) in raw_cols
])
print(f"✓ Raw validation metrics (summary-overlap only): {RAW_VALIDATION_METRIC_COLS}")

print("\n[5/6] Lightweight schema sanity checks...")
if SUMMARY_SOURCE_MODE == "percentile":
    print(f"✓ Using percentile-based summary features with preferred level: {SUMMARY_FEATURE_GROUP}")
elif SUMMARY_SOURCE_MODE == "raw_fallback":
    print("✓ Using raw-named metric fallback columns from summary table")
else:
    print("✗ Summary schema unresolved")

print("\n[6/6] Final resolved config")
print(f"  CORE_RAW_METRIC_COLS        = {CORE_RAW_METRIC_COLS}")
print(f"  AVAILABLE_EXTENDED_RAW      = {AVAILABLE_EXTENDED_RAW_METRIC_COLS}")
print(f"  AVAILABLE_INDEX_RAW         = {AVAILABLE_INDEX_RAW_METRIC_COLS}")
print(f"  SAMPLE_RAW_METRIC_COLS      = {SAMPLE_RAW_METRIC_COLS}")
print(f"  DENSITY_METRIC_COLS         = {DENSITY_METRIC_COLS}")
print(f"  RAW_VALIDATION_METRIC_COLS  = {RAW_VALIDATION_METRIC_COLS}")
print(f"  RAW_TIME_COL                = {RAW_TIME_COL}")
print(f"  SUMMARY_TIME_COL            = {SUMMARY_TIME_COL}")
print(f"  SUMMARY_FEATURE_GROUP       = {SUMMARY_FEATURE_GROUP}")
print(f"  SUMMARY_METRIC_COLS         = {SUMMARY_METRIC_COLS}")
print(f"  SUMMARY_PERCENTILE_GROUPS   = {SUMMARY_PERCENTILE_GROUPS}")
print(f"  SUMMARY_ALL_FEATURE_COLS    = {SUMMARY_ALL_FEATURE_COLS}")
print(f"  SUMMARY_SOURCE_MODE         = {SUMMARY_SOURCE_MODE}")

if not SUMMARY_TIME_COL or not SUMMARY_METRIC_COLS:
    raise ValueError("Summary schema resolution failed; inspect df_summary schema and adjust config candidates.")
if not DENSITY_METRIC_COLS:
    raise ValueError("No density metrics resolved from raw schema.")
if len(RAW_VALIDATION_METRIC_COLS) < 2:
    raise ValueError("Need at least two raw validation metrics overlapping summary features.")

print("\n" + "="*60)
print("✓ SOURCE VALIDATION COMPLETE")
print("="*60)


SOURCE DATA VALIDATION
Segment dimensions validated: ['gen', 'region_rollup', 'customer_segment', 'product_segment', 'is_training']

[1/6] Large-table row counts intentionally skipped (scan-safe)
[2/6] Resolving raw metric tiers...
✓ All core raw metrics present: ['tensor_util', 'gpu_util', 'tensor_tflops', 'tensor_tflops_sm', 'chip_power', 'redfish_power', 'sm_active']
✓ Extended raw metrics available: ['dram_active', 'mem_copy_util', 'vram_usage', 'sm_clock', 'sm_occupancy', 'TFLOPS_per_watt_efficiency']
✓ Auto-detected index metrics available: ['compute_occupancy_index', 'compute_pct_max_index', 'compute_to_memory_ratio_index', 'memory_boundness_index', 'memory_intensity_index', 'power_pct_max_index', 'tensor_to_gpu_ratio_index', 'tensor_util_dram_index']
✓ Raw sample metrics (sampled/raw exploratory path): ['tensor_util', 'gpu_util', 'tensor_tflops', 'tensor_tflops_sm', 'chip_power', 'redfish_power', 'sm_active', 'dram_active', 'mem_copy_util', 'vram_usage', 'sm_clock', 'sm_occupan

In [7]:
# ========================================
# CREATE SAMPLED RAW DATA
# ========================================
# Sample df_raw for point-level analysis (scatter plots, raw validation)
# Persist the sampled frame because it is reused across multiple analyses.
# Keep the expensive density path separate and bounded to DENSITY_METRIC_COLS.

print("="*60)
print("CREATING SAMPLED RAW DATA")
print("="*60)

print(f"\nSampling {RAW_SAMPLE_FRACTION:.04%} of raw data...")
print(f"Seed: {SAMPLE_SEED}")

# Include segment dimensions if available
segment_cols_in_raw = [dim for dim in SEGMENT_DIMS if dim in df_raw.columns]
required_cols = SAMPLE_RAW_METRIC_COLS + [RAW_TIME_COL] + segment_cols_in_raw
df_raw_selected = df_raw.select(*required_cols)

# Cast segment columns to string and fill nulls with "unknown"
for seg_col in segment_cols_in_raw:
    df_raw_selected = df_raw_selected.withColumn(seg_col, F.coalesce(F.col(seg_col).cast(T.StringType()), F.lit("unknown")))

df_raw_sample = (
    df_raw_selected
    .sample(False, RAW_SAMPLE_FRACTION, SAMPLE_SEED)
    .select(
        RAW_TIME_COL,
        *[F.col(c).cast(T.DoubleType()).alias(c) for c in SAMPLE_RAW_METRIC_COLS],
        *segment_cols_in_raw,
    )
    .na.drop(subset=CORE_RAW_METRIC_COLS)
)

from pyspark import StorageLevel
df_raw_sample = df_raw_sample.persist(StorageLevel.MEMORY_AND_DISK)

raw_sample_table = f"{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_raw_sample"
print(f"Writing sampled raw points to: {raw_sample_table}")

# Add segment columns for baseline (unsegmented)
df_raw_sample_with_segments = df_raw_sample.withColumn("segment_dim", F.lit("all")).withColumn("segment_value", F.lit("all"))

df_raw_sample_with_segments.write.mode("overwrite").saveAsTable(raw_sample_table)

print("\n✓ Sampled raw data created and persisted")
print("  Sample row count intentionally not computed (scan-safe)")
print("  Sample preview intentionally skipped (scan-safe)")
print("  Storage level: MEMORY_AND_DISK")
print(f"  Columns: {required_cols}")
print(f"  Required non-null core metrics: {CORE_RAW_METRIC_COLS}")
print("="*60)
print("NOTE: df_raw_sample is persisted and will be used for:")
print("  - Pairwise scatter plots")
print("  - Raw correlation validation")
print("  - Point-level distribution checks")
print("  - Optional exploratory index-metric analysis on the sampled path")
print("  Remember to unpersist() after last use!")
print("="*60)


CREATING SAMPLED RAW DATA

Sampling 0.0500% of raw data...
Seed: 42


Writing sampled raw points to: sandbox.sandbox_finance.dcgm_eda_raw_sample

✓ Sampled raw data created and persisted
  Sample row count intentionally not computed (scan-safe)
  Sample preview intentionally skipped (scan-safe)
  Storage level: MEMORY_AND_DISK
  Columns: ['tensor_util', 'gpu_util', 'tensor_tflops', 'tensor_tflops_sm', 'chip_power', 'redfish_power', 'sm_active', 'dram_active', 'mem_copy_util', 'vram_usage', 'sm_clock', 'sm_occupancy', 'TFLOPS_per_watt_efficiency', 'compute_occupancy_index', 'compute_pct_max_index', 'compute_to_memory_ratio_index', 'memory_boundness_index', 'memory_intensity_index', 'power_pct_max_index', 'tensor_to_gpu_ratio_index', 'tensor_util_dram_index', 'datestamp', 'gen', 'region_rollup', 'customer_segment', 'product_segment', 'is_training']
  Required non-null core metrics: ['tensor_util', 'gpu_util', 'tensor_tflops', 'tensor_tflops_sm', 'chip_power', 'redfish_power', 'sm_active']
NOTE: df_raw_sample is persisted and will be used for:
  - Pairwise 

In [8]:
# ========================================
# SUMMARY CORRELATIONS: P50 / P95 / P99
# ========================================


def get_segment_groups():
    """
    Returns a list of (segment_dim, segment_values_list) tuples.
    Includes the baseline ('all', ['all']) plus one entry per SEGMENT_DIMS dimension.
    """
    groups = [("all", ["all"])]  # baseline unsegmented
    for dim in SEGMENT_DIMS:
        groups.append((dim, None))  # None means "discover from data"
    return groups

def build_all_summary_corrs_segmented(df_summary_clean, summary_groups):
    """Build correlations for all segments"""
    import pandas as pd
    all_frames = []

    segment_groups = get_segment_groups()

    for seg_dim, seg_vals in segment_groups:
        print(f"\nProcessing segment dimension: {seg_dim}")

        if seg_dim == "all":
            # Baseline unsegmented
            seg_values_to_process = ["all"]
        else:
            # Discover segment values from data
            distinct_vals = df_summary_clean.select(seg_dim).distinct().collect()
            seg_values_to_process = [row[seg_dim] if row[seg_dim] is not None else "unknown" for row in distinct_vals]
            print(f"  Found {len(seg_values_to_process)} values for {seg_dim}")

        for seg_val in seg_values_to_process:
            # Filter data if segmented
            if seg_dim == "all":
                df_filtered = df_summary_clean
            else:
                if seg_val == "unknown":
                    df_filtered = df_summary_clean.filter(F.col(seg_dim).isNull())
                else:
                    df_filtered = df_summary_clean.filter(F.col(seg_dim) == seg_val)

            # Build correlations for this segment
            corr_pdf = build_all_summary_corrs(df_filtered, summary_groups)

            if not corr_pdf.empty:
                corr_pdf['segment_dim'] = seg_dim
                corr_pdf['segment_value'] = str(seg_val)
                all_frames.append(corr_pdf)

    if not all_frames:
        return pd.DataFrame(columns=['metric_x', 'metric_y', 'base_metric_x', 'base_metric_y',
                                     'correlation_value', 'method', 'summary_percentile',
                                     'segment_dim', 'segment_value'])

    return pd.concat(all_frames, ignore_index=True)


print("=" * 60)
print("SUMMARY CORRELATIONS: P50 / P95 / P99")
print("=" * 60)

SUMMARY_TOP_VALIDATION_PAIRS = globals().get('SUMMARY_TOP_VALIDATION_PAIRS', 5)
summary_percentile_groups = SUMMARY_PERCENTILE_GROUPS
print("Resolved summary percentile groups:")
for pct, cols in summary_percentile_groups.items():
    print(f"  {pct}: {len(cols)} metrics")

all_summary_cols = SUMMARY_ALL_FEATURE_COLS
summary_time_col_resolved = SUMMARY_TIME_COL
summary_keep_cols = [c for c in [summary_time_col_resolved] + all_summary_cols if c in df_summary.columns]

# Discover and add segment dimension columns for corr_summary segmentation
summary_segment_cols = [c for c in SEGMENT_DIMS if c in df_summary.columns and c not in summary_keep_cols]
if summary_segment_cols:
    print(f"✅ Adding segment columns to df_summary_clean: {summary_segment_cols}")
    summary_keep_cols = summary_keep_cols + summary_segment_cols
else:
    print("⚠️  No segment columns found in df_summary - corr_summary will be baseline-only")

print(f"ℹ️  summary_keep_cols final count: {len(summary_keep_cols)} (time + metrics + segments)")

from pyspark import StorageLevel
df_summary_clean = df_summary.select(*summary_keep_cols).persist(StorageLevel.MEMORY_AND_DISK)

# Validate that segment columns are present in df_summary_clean
available_seg_cols = [c for c in SEGMENT_DIMS if c in df_summary_clean.columns]
print(f"✅ Segment columns available in df_summary_clean: {available_seg_cols}")

# Show sample segment values for validation (bounded collection)
if available_seg_cols:
    print("\nSegment dimension sample values:")
    for seg_dim in available_seg_cols[:2]:  # Show first 2 dimensions only
        sample_vals = [r[0] for r in df_summary_clean.select(seg_dim).where(F.col(seg_dim).isNotNull()).distinct().limit(5).collect()]
        print(f"  {seg_dim}: {sample_vals[:5]}")

corr_summary_pdf = build_all_summary_corrs_segmented(df_summary_clean, summary_percentile_groups)

if corr_summary_pdf.empty:
    print("No summary correlation rows generated.")
    auto_validation_pairs = []
else:
    corr_summary_pdf['run_id'] = RUN_ID
    corr_summary_pdf['run_timestamp'] = pd.Timestamp(RUN_TIMESTAMP)
    corr_summary_pdf['summary_time_col'] = summary_time_col_resolved
    corr_summary_pdf['obs_count'] = None
    corr_summary_pdf['summary_feature_group'] = corr_summary_pdf['summary_percentile']
    print(f"Generated {len(corr_summary_pdf)} summary-correlation rows across percentile tiers.")

    # Validate segmented corr_summary output
    print("\n" + "="*60)
    print("CORR_SUMMARY SEGMENTATION VALIDATION")
    print("="*60)
    print(f"Total correlation rows: {len(corr_summary_pdf)}")
    
    # Check for segment columns
    if 'segment_dim' in corr_summary_pdf.columns and 'segment_value' in corr_summary_pdf.columns:
        print("✅ segment_dim and segment_value columns present")
        
        # Show baseline rows
        baseline_count = len(corr_summary_pdf[(corr_summary_pdf['segment_dim'] == 'all') & (corr_summary_pdf['segment_value'] == 'all')])
        print(f"  Baseline rows (all/all): {baseline_count}")
        
        # Show segmented rows
        seg_dims_present = corr_summary_pdf[corr_summary_pdf['segment_dim'] != 'all']['segment_dim'].unique()
        print(f"  Segmented dimensions: {list(seg_dims_present)}")
        
        for seg_dim in list(seg_dims_present)[:2]:  # Show first 2 dimensions
            seg_count = len(corr_summary_pdf[corr_summary_pdf['segment_dim'] == seg_dim])
            seg_vals = corr_summary_pdf[corr_summary_pdf['segment_dim'] == seg_dim]['segment_value'].unique()
            print(f"  {seg_dim}: {seg_count} rows across {len(seg_vals)} values - {list(seg_vals[:3])}")
    else:
        print("❌ WARNING: segment_dim or segment_value missing from corr_summary_pdf")
    
    print("="*60)
    auto_validation_pairs = build_auto_validation_pairs(corr_summary_pdf, top_n=SUMMARY_TOP_VALIDATION_PAIRS)


SUMMARY CORRELATIONS: P50 / P95 / P99
Resolved summary percentile groups:
  p50: 15 metrics
  p95: 15 metrics
  p99: 15 metrics
✅ Adding segment columns to df_summary_clean: ['gen', 'region_rollup', 'customer_segment', 'product_segment', 'is_training']
ℹ️  summary_keep_cols final count: 51 (time + metrics + segments)
✅ Segment columns available in df_summary_clean: ['gen', 'region_rollup', 'customer_segment', 'product_segment', 'is_training']

Segment dimension sample values:
  gen: ['current_gen', 'prior_gen']
  region_rollup: ['NAM', 'EU']

Processing segment dimension: all

Processing segment dimension: gen
  Found 2 values for gen

Processing segment dimension: region_rollup
  Found 2 values for region_rollup

Processing segment dimension: customer_segment
  Found 4 values for customer_segment

Processing segment dimension: product_segment
  Found 4 values for product_segment

Processing segment dimension: is_training
  Found 2 values for is_training
Generated 8658 summary-correlat

In [9]:
# ========================================
# RAW-LEVEL CORRELATION VALIDATION
# ========================================
# Validate summary correlations using sampled raw data on the overlapping base metrics only.

print("="*60)
print("RAW-LEVEL CORRELATION VALIDATION")
print("="*60)

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

print("\nUsing df_raw_sample (row count intentionally skipped to avoid full scan)")
print("Comparing raw-level correlations against summary-level feature correlations...")
print(f"Using overlapping raw validation metrics: {RAW_VALIDATION_METRIC_COLS}")

raw_corr_results = {}
for method in CORR_METHODS:
    print(f"\nComputing {method.upper()} correlation on raw sample...")
    vec_assembler = VectorAssembler(inputCols=RAW_VALIDATION_METRIC_COLS, outputCol="features", handleInvalid="skip")
    vec_df = vec_assembler.transform(df_raw_sample.select(*RAW_VALIDATION_METRIC_COLS)).select("features")
    raw_corr_matrix = Correlation.corr(vec_df, "features", method=method).head()[0].toArray()
    raw_corr_results[method] = raw_corr_matrix
    print(f"✓ {method.upper()} raw correlation computed")

validation_rows = []
for pair in auto_validation_pairs:
    raw_x = pair['base_metric_x']
    raw_y = pair['base_metric_y']
    method = pair['method']
    if raw_x in RAW_VALIDATION_METRIC_COLS and raw_y in RAW_VALIDATION_METRIC_COLS and method in raw_corr_results:
        ri = RAW_VALIDATION_METRIC_COLS.index(raw_x)
        rj = RAW_VALIDATION_METRIC_COLS.index(raw_y)
        raw_corr = float(raw_corr_results[method][ri][rj])
        diff = abs(float(pair['summary_correlation']) - raw_corr)
        validation_rows.append({
            'summary_percentile': pair['summary_percentile'],
            'method': method,
            'summary_metric_x': pair['summary_metric_x'],
            'summary_metric_y': pair['summary_metric_y'],
            'base_metric_x': raw_x,
            'base_metric_y': raw_y,
            'summary_correlation': float(pair['summary_correlation']),
            'raw_correlation': raw_corr,
            'abs_diff': diff,
        })

validation_pdf = pd.DataFrame(validation_rows)

print("\n" + "="*60)
print("CORRELATION COMPARISON: Summary vs Raw")
print("="*60)
if validation_pdf.empty:
    print("No overlapping auto-validation pairs available.")
else:
    for method in CORR_METHODS:
        method_pdf = validation_pdf[validation_pdf['method'] == method].copy()
        if method_pdf.empty:
            continue
        print(f"\n{method.upper()} - Auto-selected validation pairs:")
        print(f"{'Percentile':<12} {'Summary Pair':<48} {'Summary':>10} {'Raw':>10} {'Diff':>10}")
        print("-" * 96)
        for _, row in method_pdf.sort_values(['summary_percentile', 'abs_diff']).iterrows():
            label = f"{row['summary_metric_x']} [{row['base_metric_x']}] <-> {row['summary_metric_y']} [{row['base_metric_y']}]"
            print(f"{row['summary_percentile']:<12} {label:<48} {row['summary_correlation']:>10.3f} {row['raw_correlation']:>10.3f} {row['abs_diff']:>10.3f}")

print("\n" + "="*60)
print("VALIDATION ANALYSIS")
print("="*60)
if validation_pdf.empty:
    print("No validation pairs available.")
else:
    for method in CORR_METHODS:
        method_pdf = validation_pdf[validation_pdf['method'] == method]
        if method_pdf.empty:
            continue
        print(f"\n{method.upper()}:")
        print(f"  Mean absolute difference: {method_pdf['abs_diff'].mean():.4f}")
        print(f"  Max absolute difference:  {method_pdf['abs_diff'].max():.4f}")
        print("  Differences are expected due to:")
        print("    - Aggregation effects in summary data")
        print("    - Sampling variance in raw data")
        print("    - Different observation grains")

print("\n" + "="*60)
print("✓ RAW CORRELATION VALIDATION COMPLETE")
print("="*60)


RAW-LEVEL CORRELATION VALIDATION

Using df_raw_sample (row count intentionally skipped to avoid full scan)
Comparing raw-level correlations against summary-level feature correlations...
Using overlapping raw validation metrics: ['tensor_util', 'gpu_util', 'tensor_tflops_sm', 'chip_power', 'redfish_power', 'sm_active', 'dram_active', 'mem_copy_util', 'vram_usage', 'sm_clock', 'sm_occupancy', 'TFLOPS_per_watt_efficiency', 'compute_occupancy_index', 'memory_boundness_index', 'tensor_util_dram_index']

Computing PEARSON correlation on raw sample...
✓ PEARSON raw correlation computed

Computing SPEARMAN correlation on raw sample...
✓ SPEARMAN raw correlation computed

CORRELATION COMPARISON: Summary vs Raw

PEARSON - Auto-selected validation pairs:
Percentile   Summary Pair                                        Summary        Raw       Diff
------------------------------------------------------------------------------------------------
p50          dram_active_p50 [dram_active] <-> mem_cop

In [10]:
# ========================================
# CORRELATION OVER TIME: P50 / P95 / P99 (WITH SEGMENTATION)
# ========================================

corr_over_time_frames = []
summary_time_col_resolved = SUMMARY_TIME_COL

segment_groups = get_segment_groups()

for seg_dim, seg_vals in segment_groups:
    print(f"\nProcessing correlation over time for segment dimension: {seg_dim}")

    if seg_dim == "all":
        seg_values_to_process = ["all"]
    else:
        distinct_vals = df_summary.select(seg_dim).distinct().collect()
        seg_values_to_process = [row[seg_dim] if row[seg_dim] is not None else "unknown" for row in distinct_vals]
        print(f"  Found {len(seg_values_to_process)} values for {seg_dim}")

    for seg_val in seg_values_to_process:
        # Filter data if segmented
        if seg_dim == "all":
            df_seg = df_summary
        else:
            if seg_val == "unknown":
                df_seg = df_summary.filter(F.col(seg_dim).isNull())
            else:
                df_seg = df_summary.filter(F.col(seg_dim) == seg_val)

        # Compute correlations over time for this segment
        for pct, cols in SUMMARY_PERCENTILE_GROUPS.items():
            if len(cols) < 2:
                continue
            time_pdf = df_seg.select(*([summary_time_col_resolved] + cols)).toPandas()
            time_pdf[summary_time_col_resolved] = pd.to_datetime(time_pdf[summary_time_col_resolved], errors='coerce')
            time_pdf = time_pdf.dropna(subset=[summary_time_col_resolved])
            if time_pdf.empty:
                continue
            time_pdf['time_bucket'] = time_pdf[summary_time_col_resolved].dt.floor('D')
            for bucket, bucket_pdf in time_pdf.groupby('time_bucket'):
                if bucket_pdf.shape[0] < 3:
                    continue
                metrics_pdf = bucket_pdf[cols].apply(pd.to_numeric, errors='coerce')
                for method in CORR_METHODS:
                    corr = metrics_pdf.corr(method=method)
                    ccols = list(corr.columns)
                    for i, x in enumerate(ccols):
                        for j, y in enumerate(ccols):
                            if i >= j:
                                continue
                            corr_over_time_frames.append({
                                'time_bucket': bucket,
                                'metric_x': x,
                                'metric_y': y,
                                'base_metric_x': normalize_metric_name(x),
                                'base_metric_y': normalize_metric_name(y),
                                'correlation_value': float(corr.loc[x, y]),
                                'method': method,
                                'summary_percentile': pct,
                                'run_id': RUN_ID,
                                'time_grain': 'day',
                                'segment_dim': seg_dim,
                                'segment_value': str(seg_val),
                            })

corr_over_time_pdf = pd.DataFrame(corr_over_time_frames)
print(f"Generated {len(corr_over_time_pdf)} correlation-over-time rows.")




Processing correlation over time for segment dimension: all

Processing correlation over time for segment dimension: gen
  Found 2 values for gen

Processing correlation over time for segment dimension: region_rollup
  Found 2 values for region_rollup

Processing correlation over time for segment dimension: customer_segment
  Found 4 values for customer_segment

Processing correlation over time for segment dimension: product_segment
  Found 4 values for product_segment

Processing correlation over time for segment dimension: is_training
  Found 2 values for is_training
Generated 4241790 correlation-over-time rows.


In [11]:
# ========================================
# PERSIST CORRELATION OUTPUTS
# ========================================

print("=" * 60)
print("PERSISTING CORRELATION OUTPUTS")
print("=" * 60)

corr_summary_schema = T.StructType([
    T.StructField("metric_x", T.StringType(), False),
    T.StructField("metric_y", T.StringType(), False),
    T.StructField("base_metric_x", T.StringType(), True),
    T.StructField("base_metric_y", T.StringType(), True),
    T.StructField("method", T.StringType(), False),
    T.StructField("summary_percentile", T.StringType(), True),
    T.StructField("summary_feature_group", T.StringType(), True),
    T.StructField("correlation_value", T.DoubleType(), True),
    T.StructField("obs_count", T.LongType(), True),
    T.StructField("run_id", T.StringType(), False),
    T.StructField("run_timestamp", T.TimestampType(), True),
    T.StructField("summary_time_col", T.StringType(), True),
    T.StructField("segment_dim", T.StringType(), False),
    T.StructField("segment_value", T.StringType(), False),
])

corr_over_time_schema = T.StructType([
    T.StructField("time_bucket", T.TimestampType(), True),
    T.StructField("metric_x", T.StringType(), False),
    T.StructField("metric_y", T.StringType(), False),
    T.StructField("base_metric_x", T.StringType(), True),
    T.StructField("base_metric_y", T.StringType(), True),
    T.StructField("correlation_value", T.DoubleType(), True),
    T.StructField("method", T.StringType(), False),
    T.StructField("summary_percentile", T.StringType(), True),
    T.StructField("run_id", T.StringType(), False),
    T.StructField("time_grain", T.StringType(), True),
    T.StructField("segment_dim", T.StringType(), False),
    T.StructField("segment_value", T.StringType(), False),
])

print("\n[1/3] Creating correlation summary table...")
if corr_summary_pdf.empty:
    corr_summary_df = spark.createDataFrame([], schema=corr_summary_schema)
else:
    tmp = corr_summary_pdf.copy()
    if 'obs_count' not in tmp.columns:
        tmp['obs_count'] = -1
    tmp['obs_count'] = pd.to_numeric(tmp['obs_count'], errors='coerce').fillna(-1).astype('int64')
    if 'run_timestamp' not in tmp.columns:
        tmp['run_timestamp'] = pd.Timestamp(RUN_TIMESTAMP)
    tmp['run_timestamp'] = pd.to_datetime(tmp['run_timestamp'], errors='coerce')
    tmp['correlation_value'] = pd.to_numeric(tmp['correlation_value'], errors='coerce')
    for col in ['metric_x', 'metric_y', 'base_metric_x', 'base_metric_y', 'method', 'summary_percentile', 'summary_feature_group', 'run_id', 'summary_time_col', 'segment_dim', 'segment_value']:
        if col in tmp.columns:
            tmp[col] = tmp[col].astype('string')
    
    # Convert to records and scrub timestamps
    records = tmp.to_dict('records')
    for record in records:
        for col in ['run_timestamp']:
            if col in record:
                if pd.isna(record[col]):
                    record[col] = None
                elif isinstance(record[col], pd.Timestamp):
                    record[col] = record[col].to_pydatetime()
    
    corr_summary_df = spark.createDataFrame(records, schema=corr_summary_schema)

corr_summary_table = f"{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_corr_summary"
print(f"Writing to: {corr_summary_table}")
corr_summary_df.write.mode("overwrite").saveAsTable(corr_summary_table)

print("\n[2/3] Writing correlation over time table...")
if corr_over_time_pdf.empty:
    corr_over_time_df = spark.createDataFrame([], schema=corr_over_time_schema)
else:
    tmp = corr_over_time_pdf.copy()
    tmp['time_bucket'] = pd.to_datetime(tmp['time_bucket'], errors='coerce')
    tmp['correlation_value'] = pd.to_numeric(tmp['correlation_value'], errors='coerce')
    for col in ['metric_x', 'metric_y', 'base_metric_x', 'base_metric_y', 'method', 'summary_percentile', 'run_id', 'time_grain', 'segment_dim', 'segment_value']:
        if col in tmp.columns:
            tmp[col] = tmp[col].astype('string')
    
    # Convert to records and scrub timestamps
    records = tmp.to_dict('records')
    for record in records:
        for col in ['time_bucket']:
            if col in record:
                if pd.isna(record[col]):
                    record[col] = None
                elif isinstance(record[col], pd.Timestamp):
                    record[col] = record[col].to_pydatetime()
    
    corr_over_time_df = spark.createDataFrame(records, schema=corr_over_time_schema)

corr_over_time_table = f"{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_corr_over_time"
print(f"Writing to: {corr_over_time_table}")
corr_over_time_df.write.mode("overwrite").saveAsTable(corr_over_time_table)

print("\n[3/3] Cleaning up cached DataFrames...")
df_summary_clean.unpersist()
print("✓ Unpersisted df_summary_clean")

print("\n" + "=" * 60)
print("✓ CORRELATION OUTPUTS PERSISTED")
print("=" * 60)
print("\nCreated tables:")
print(f"  1. {corr_summary_table}")
print(f"  2. {corr_over_time_table}")
print("=" * 60)


PERSISTING CORRELATION OUTPUTS

[1/3] Creating correlation summary table...
Writing to: sandbox.sandbox_finance.dcgm_eda_corr_summary

[2/3] Writing correlation over time table...
Writing to: sandbox.sandbox_finance.dcgm_eda_corr_over_time

[3/3] Cleaning up cached DataFrames...
✓ Unpersisted df_summary_clean

✓ CORRELATION OUTPUTS PERSISTED

Created tables:
  1. sandbox.sandbox_finance.dcgm_eda_corr_summary
  2. sandbox.sandbox_finance.dcgm_eda_corr_over_time


In [12]:
# ========================================
# DENSITY MATRIX FUNCTIONS (FIXED BINS)
# ========================================
# Key change: Compute GLOBAL bin edges once, reuse across all time periods

from typing import List, Dict, Tuple
from pyspark import StorageLevel
from pyspark.ml.feature import Bucketizer


def compute_global_bins(sdf, cols: List[str], bins: int = 64, percentile_range: Tuple[float, float] = (0.01, 0.99)):
    """
    Compute FIXED global bin edges for all specified columns.
    These bins will be reused across all time periods for comparability.

    Returns:
        splits_dict: Dict[col_name, List[float]] - bin edges for each column
        bin_metadata_df: DataFrame with bin metadata for persistence
    """
    print(f"\n[compute_global_bins] Computing fixed bins for {len(cols)} columns...")
    print(f"  Percentile range: {percentile_range}")
    print(f"  Bins per column: {bins}")

    pct_low, pct_high = percentile_range
    agg_exprs = [
        F.expr(f"percentile_approx({col}, array({pct_low}, {pct_high}), 1000)").alias(col)
        for col in cols
    ]

    range_row = sdf.agg(*agg_exprs).first()

    splits_dict = {}
    bin_metadata_rows = []

    for col in cols:
        pcts = range_row[col]
        
        # Handle None percentiles (empty or all-NULL data for this column)
        if pcts is None or pcts[0] is None or pcts[1] is None:
            print(f"  ⚠️  {col}: percentile_approx returned {pcts} - using default range [0, 1]")
            vmin, vmax = 0.0, 1.0
        else:
            vmin, vmax = float(pcts[0]), float(pcts[1])

        span = vmax - vmin
        if span == 0:
            span = 1.0
        vmin -= span * 0.001
        vmax += span * 0.001

        splits_list = [-float('inf')] + list(np.linspace(vmin, vmax, bins + 1)) + [float('inf')]
        splits_dict[col] = splits_list

        finite_edges = splits_list[1:-1]
        for bin_num, (bin_min, bin_max) in enumerate(zip(finite_edges[:-1], finite_edges[1:])):
            bin_metadata_rows.append({
                "metric_name": col,
                "bin_number": bin_num,
                "bin_min": float(bin_min),
                "bin_max": float(bin_max),
                "num_bins": bins,
                "percentile_low": pct_low,
                "percentile_high": pct_high,
                "run_id": RUN_ID,
                "segment_dim": "all",
                "segment_value": "all"
            })

        if pcts is not None and pcts[0] is not None and pcts[1] is not None:
            print(f"  {col}: [{vmin:.4f}, {vmax:.4f}] -> {bins} uniform bins")

    bin_metadata_schema = T.StructType([
        T.StructField("metric_name", T.StringType(), False),
        T.StructField("bin_number", T.IntegerType(), False),
        T.StructField("bin_min", T.DoubleType(), False),
        T.StructField("bin_max", T.DoubleType(), False),
        T.StructField("num_bins", T.IntegerType(), False),
        T.StructField("percentile_low", T.DoubleType(), False),
        T.StructField("percentile_high", T.DoubleType(), False),
        T.StructField("run_id", T.StringType(), False),
        T.StructField("segment_dim", T.StringType(), False),
        T.StructField("segment_value", T.StringType(), False)
    ])

    bin_metadata_df = spark.createDataFrame(bin_metadata_rows, schema=bin_metadata_schema)

    print(f"✓ Global bins computed")
    return splits_dict, bin_metadata_df


def bucketize_all_columns(sdf, cols: List[str], splits_dict: Dict[str, List[float]]):
    """Apply Bucketizer to all columns once; returns binned DataFrame."""
    sdf_binned = sdf
    for col in cols:
        bucketizer = Bucketizer(
            splits=splits_dict[col],
            inputCol=col,
            outputCol=f"{col}_bin",
            handleInvalid="keep"
        )
        sdf_binned = bucketizer.transform(sdf_binned)
        sdf_binned = sdf_binned.withColumn(
            f"{col}_bin",
            (F.col(f"{col}_bin").cast("int") - F.lit(1))
        )
    return sdf_binned


def compute_density_pairs_df(sdf_binned, cols: List[str], num_bins: int, include_time_bucket: bool = False):
    """
    Compute 2D density histograms for all pairs in Spark (no collect).
    Returns a single DataFrame with columns: metric_x, metric_y, x_bin, y_bin, count, [time_bucket]
    """
    pair_dfs = []
    for i, col_x in enumerate(cols):
        for j, col_y in enumerate(cols):
            if i >= j:
                continue
            bin_x = f"{col_x}_bin"
            bin_y = f"{col_y}_bin"

            base = sdf_binned
            if include_time_bucket:
                group_cols = ["time_bucket", bin_x, bin_y]
                select_cols = ["time_bucket", bin_x, bin_y]
            else:
                group_cols = [bin_x, bin_y]
                select_cols = [bin_x, bin_y]

            df_pair = (
                base
                .select(*select_cols)
                .filter((F.col(bin_x) >= 0) & (F.col(bin_x) < num_bins) &
                        (F.col(bin_y) >= 0) & (F.col(bin_y) < num_bins))
                .groupBy(*group_cols)
                .count()
                .withColumn("metric_x", F.lit(col_x))
                .withColumn("metric_y", F.lit(col_y))
                .withColumnRenamed(bin_x, "x_bin")
                .withColumnRenamed(bin_y, "y_bin")
            )

            if include_time_bucket:
                df_pair = df_pair.select("time_bucket", "metric_x", "metric_y", "x_bin", "y_bin", "count")
            else:
                df_pair = df_pair.select("metric_x", "metric_y", "x_bin", "y_bin", "count")

            pair_dfs.append(df_pair)

    if not pair_dfs:
        return None

    # Union all pairs
    out = pair_dfs[0]
    for df in pair_dfs[1:]:
        out = out.unionByName(df)
    return out


print("="*60)
print("DENSITY MATRIX FUNCTIONS LOADED")
print("="*60)
print("✓ compute_global_bins() - Computes fixed bin edges")
print("✓ bucketize_all_columns() - Bucketize all columns once")
print("✓ compute_density_pairs_df() - Spark-side 2D histograms")
print("="*60)


DENSITY MATRIX FUNCTIONS LOADED
✓ compute_global_bins() - Computes fixed bin edges
✓ bucketize_all_columns() - Bucketize all columns once
✓ compute_density_pairs_df() - Spark-side 2D histograms


In [13]:
# ========================================
# HOLOVIEWS RENDERING FUNCTIONS
# ========================================
# Render density matrix from density results using HoloViews
# NOTE: Imports moved inside function to avoid import errors when packages aren't yet available

def render_density_from_results(density_results, splits_dict, cols, cmap="fire", img_size=400):
    """
    Render density matrix from density results using HoloViews.

    Args:
        density_results: List[Dict] from compute_density_matrix_fixed_bins()
        splits_dict: Bin edges dictionary
        cols: List of column names
        cmap: Colormap name
        img_size: Size of each plot in pixels
    """
    # Import visualization libraries here (lazy import after packages are installed)
    import matplotlib
    matplotlib.use("Agg")  # Non-interactive backend for notebook
    import matplotlib.pyplot as plt
    import holoviews as hv
    import panel as pn
    
    hv.extension('bokeh')
    pn.extension()
    
    print(f"Rendering {len(cols)}x{len(cols)} density matrix...")

    # Convert results list to lookup dict
    results_dict = {}
    for row in density_results:
        key = (row["metric_x"], row["metric_y"])
        if key not in results_dict:
            results_dict[key] = []
        results_dict[key].append(row)

    plots = []
    n_cols = len(cols)

    for i, col_x in enumerate(cols):
        row_plots = []
        print(f"  Rendering row {i+1}/{n_cols}: {col_x}")

        for j, col_y in enumerate(cols):
            if i == j:
                # Diagonal: 1D histogram (simplified for now)
                row_plots.append(hv.Empty().opts(width=img_size//2, height=img_size//2))
            else:
                # Off-diagonal: 2D heatmap
                key = (col_x, col_y)
                hist_2d = results_dict.get(key, [])

                if not hist_2d:
                    row_plots.append(hv.Empty().opts(width=img_size//2, height=img_size//2))
                    continue

                # Get bin edges
                edges_x = np.array(splits_dict[col_x])
                edges_y = np.array(splits_dict[col_y])

                x_finite = edges_x[(edges_x != -np.inf) & (edges_x != np.inf)]
                y_finite = edges_y[(edges_y != -np.inf) & (edges_y != np.inf)]

                nx = len(x_finite) - 1
                ny = len(y_finite) - 1

                # Build 2D array
                img = np.zeros((ny, nx), dtype=np.float32)
                for row in hist_2d:
                    x_bin = row["x_bin"]
                    y_bin = row["y_bin"]
                    count = row["count"]
                    if 0 <= x_bin < nx and 0 <= y_bin < ny:
                        img[y_bin, x_bin] = count

                # Handle zeros for log scale
                img[img == 0] = np.nan
                img = np.flipud(img)

                if not np.any(np.isfinite(img)):
                    row_plots.append(hv.Empty().opts(width=img_size//2, height=img_size//2))
                    continue

                # Color limits
                vmin = 1.0
                finite_vals = img[np.isfinite(img)]
                vmax = float(np.percentile(finite_vals, 99.5)) if len(finite_vals) > 0 else 1.0

                # Create HoloViews Image
                bounds = (x_finite[0], y_finite[0], x_finite[-1], y_finite[-1])
                image = hv.Image(img, bounds=bounds, kdims=[col_x, col_y], vdims=['count']).opts(
                    cmap=cmap,
                    colorbar=True,
                    width=img_size,
                    height=img_size,
                    logz=True,
                    clim=(vmin, vmax),
                    clipping_colors={'NaN': (0, 0, 0, 0)},
                    tools=['hover'],
                    xlabel=col_x,
                    ylabel=col_y,
                    fontsize={'xlabel': 11, 'ylabel': 11}
                )
                row_plots.append(image)

        plots.append(hv.Layout(row_plots).cols(n_cols))

    print("✓ Rendering complete")
    return hv.Layout(plots).cols(1)


print("="*60)
print("RENDERING FUNCTIONS LOADED")
print("="*60)
print("✓ render_density_from_results() - Renders HoloViews density matrix")
print("  NOTE: Visualization libraries imported lazily inside function")
print("="*60)


RENDERING FUNCTIONS LOADED
✓ render_density_from_results() - Renders HoloViews density matrix
  NOTE: Visualization libraries imported lazily inside function


In [ ]:
# ========================================
# STATIC DENSITY MATRIX EXECUTION (WITH SEGMENTATION)
# ========================================

print("="*60)
print("STATIC DENSITY MATRIX COMPUTATION")
print("="*60)
print(f"Using bounded density metric set: {DENSITY_METRIC_COLS}")

# Precompute segment values once (avoid repeated distinct().collect())
segment_values_map = {"all": ["all"]}
for dim in SEGMENT_DIMS:
    if dim in df_raw.columns:
        vals = [row[dim] if row[dim] is not None else "unknown" for row in df_raw.select(dim).distinct().collect()]
        segment_values_map[dim] = vals

# Cache a narrow base frame for density work
from pyspark import StorageLevel
_density_base_cols = [c for c in [RAW_TIME_COL] + DENSITY_METRIC_COLS + SEGMENT_DIMS if c in df_raw.columns]
_density_base = (
    df_raw
    .select(*_density_base_cols) 
    .select(
        *([RAW_TIME_COL] if RAW_TIME_COL in _density_base_cols else []),
        *[F.col(c).cast(T.DoubleType()).alias(c) for c in DENSITY_METRIC_COLS],
        *[c for c in SEGMENT_DIMS if c in _density_base_cols]
    )
)
_density_base = _density_base.persist(StorageLevel.MEMORY_AND_DISK)

all_static_density_dfs = []
all_bin_metadata_dfs = []

segment_groups = get_segment_groups()

# Cache bins per segment so time-density can reuse them
bins_cache = {}

for seg_dim, seg_vals in segment_groups:
    print(f"\nProcessing static density for segment dimension: {seg_dim}")

    if seg_dim == "all":
        seg_values_to_process = ["all"]
    else:
        seg_values_to_process = segment_values_map.get(seg_dim, [])
        print(f"  Found {len(seg_values_to_process)} values for {seg_dim}")

    for seg_val in seg_values_to_process:
        print(f"  Processing segment value: {seg_val}")

        # Filter data if segmented
        if seg_dim == "all":
            df_seg = _density_base
        else:
            if seg_val == "unknown":
                df_seg = _density_base.filter(F.col(seg_dim).isNull())
            else:
                df_seg = _density_base.filter(F.col(seg_dim) == seg_val)

        # Prepare raw data - cast already applied, now filter NULLs and NaNs
        _df_raw_for_density = df_seg.select(*DENSITY_METRIC_COLS)

        # Filter out NULLs and NaNs
        nan_filter = None
        for c in DENSITY_METRIC_COLS:
            cond = F.col(c).isNotNull() & (~F.isnan(F.col(c)))
            nan_filter = cond if nan_filter is None else (nan_filter & cond)

        _df_raw_for_density = _df_raw_for_density.filter(nan_filter)

        # Check if segment has valid data (avoid full count scan)
        has_rows = len(_df_raw_for_density.take(1)) > 0
        if not has_rows:
            print(f"    ⚠️  Skipping {seg_dim}={seg_val}: no valid rows for density")
            continue

        # Compute bins for this segment (cache for reuse)
        print("    [Step 1/3] Computing bins...")
        splits_dict_seg, bin_metadata_df_seg = compute_global_bins(
            _df_raw_for_density, DENSITY_METRIC_COLS, bins=NUM_BINS, percentile_range=PERCENTILE_RANGE
        )
        bins_cache[(seg_dim, str(seg_val))] = splits_dict_seg

        # Update bin metadata with segment info
        bin_metadata_df_seg = bin_metadata_df_seg.withColumn("segment_dim", F.lit(seg_dim))
        bin_metadata_df_seg = bin_metadata_df_seg.withColumn("segment_value", F.lit(str(seg_val)))
        all_bin_metadata_dfs.append(bin_metadata_df_seg)

        print("    [Step 2/3] Bucketizing and caching...")
        num_bins = len(splits_dict_seg[DENSITY_METRIC_COLS[0]]) - 3
        _df_binned = bucketize_all_columns(_df_raw_for_density, DENSITY_METRIC_COLS, splits_dict_seg)
        _df_binned = _df_binned.persist(StorageLevel.MEMORY_AND_DISK)

        print("    [Step 3/3] Computing density matrix...")
        density_df_seg = compute_density_pairs_df(_df_binned, DENSITY_METRIC_COLS, num_bins, include_time_bucket=False)

        # Add segment columns
        density_df_seg = density_df_seg.withColumn("segment_dim", F.lit(seg_dim))
        density_df_seg = density_df_seg.withColumn("segment_value", F.lit(str(seg_val)))
        all_static_density_dfs.append(density_df_seg)

        # Unpersist to free memory
        _df_binned.unpersist()

# Union all segment results
from functools import reduce
static_density_df = reduce(lambda a, b: a.unionByName(b), all_static_density_dfs)
static_bin_metadata_df = reduce(lambda a, b: a.unionByName(b), all_bin_metadata_dfs)

print("\n" + "="*60)
print("✓ STATIC DENSITY MATRIX COMPLETE (ALL SEGMENTS)")
print("="*60)


STATIC DENSITY MATRIX COMPUTATION
Using bounded density metric set: ['tensor_util', 'gpu_util', 'tensor_tflops', 'tensor_tflops_sm', 'chip_power', 'redfish_power', 'sm_active']

Processing static density for segment dimension: all
  Processing segment value: all
    [Step 1/3] Computing bins...

[compute_global_bins] Computing fixed bins for 7 columns...
  Percentile range: (0.01, 0.99)
  Bins per column: 64
  tensor_util: [-0.0009, 0.8750] -> 64 uniform bins
  gpu_util: [-0.0010, 1.0010] -> 64 uniform bins
  tensor_tflops: [-16.0511, 16067.1061] -> 64 uniform bins
  tensor_tflops_sm: [-12.7659, 12778.7107] -> 64 uniform bins
  chip_power: [281.4101, 7540.8229] -> 64 uniform bins
  redfish_power: [357.2750, 10299.9228] -> 64 uniform bins
  sm_active: [-0.0010, 0.9690] -> 64 uniform bins
✓ Global bins computed
    [Step 2/3] Bucketizing and caching...
    [Step 3/3] Computing density matrix...

Processing static density for segment dimension: gen
  Found 2 values for gen
  Processing se

In [15]:
# ========================================
# DENSITY OVER TIME (FIXED BINS, WITH SEGMENTATION)
# ========================================

print("="*60)
print("DENSITY OVER TIME COMPUTATION")
print("="*60)
print(f"Using bounded density metric set: {DENSITY_METRIC_COLS}")

all_density_time_dfs = []

segment_groups = get_segment_groups()

for seg_dim, seg_vals in segment_groups:
    print(f"\nProcessing density over time for segment dimension: {seg_dim}")

    if seg_dim == "all":
        seg_values_to_process = ["all"]
    else:
        seg_values_to_process = segment_values_map.get(seg_dim, [])
        print(f"  Found {len(seg_values_to_process)} values for {seg_dim}")

    for seg_val in seg_values_to_process:
        print(f"  Processing segment value: {seg_val}")

        # Filter data if segmented
        if seg_dim == "all":
            df_seg = _density_base
        else:
            if seg_val == "unknown":
                df_seg = _density_base.filter(F.col(seg_dim).isNull())
            else:
                df_seg = _density_base.filter(F.col(seg_dim) == seg_val)

        # Add time bucket and prepare data - cast already applied
        _df_raw_with_bucket = (
            df_seg
            .select(RAW_TIME_COL, *DENSITY_METRIC_COLS)
            .withColumn("time_bucket", F.date_trunc(TIME_BUCKET_GRAIN, F.to_timestamp(F.col(RAW_TIME_COL))))
            .select("time_bucket", *DENSITY_METRIC_COLS)
        )

        # Filter out NULLs and NaNs
        nan_filter = F.col("time_bucket").isNotNull()
        for c in DENSITY_METRIC_COLS:
            cond = F.col(c).isNotNull() & (~F.isnan(F.col(c)))
            nan_filter = nan_filter & cond

        _df_raw_with_bucket = _df_raw_with_bucket.filter(nan_filter)

        # Check if segment has valid data (avoid full count scan)
        has_rows = len(_df_raw_with_bucket.take(1)) > 0
        if not has_rows:
            print(f"    ⚠️  Skipping {seg_dim}={seg_val}: no valid rows for density over time")
            continue

        # Reuse bins from static density when available; otherwise compute and cache
        bins_key = (seg_dim, str(seg_val))
        splits_dict_seg = bins_cache.get(bins_key)
        if splits_dict_seg is None:
            _df_for_bins = _df_raw_with_bucket.select(*DENSITY_METRIC_COLS)
            splits_dict_seg, _ = compute_global_bins(
                _df_for_bins, DENSITY_METRIC_COLS, bins=NUM_BINS, percentile_range=PERCENTILE_RANGE
            )
            bins_cache[bins_key] = splits_dict_seg

        num_bins = len(splits_dict_seg[DENSITY_METRIC_COLS[0]]) - 3
        _df_binned_time = bucketize_all_columns(_df_raw_with_bucket, DENSITY_METRIC_COLS, splits_dict_seg)
        _df_binned_time = _df_binned_time.persist(StorageLevel.MEMORY_AND_DISK)

        density_time_df_seg = compute_density_pairs_df(_df_binned_time, DENSITY_METRIC_COLS, num_bins, include_time_bucket=True)

        # Add segment columns
        density_time_df_seg = density_time_df_seg.withColumn("segment_dim", F.lit(seg_dim))
        density_time_df_seg = density_time_df_seg.withColumn("segment_value", F.lit(str(seg_val)))
        all_density_time_dfs.append(density_time_df_seg)

        # Unpersist
        _df_binned_time.unpersist()

# Union all segment results
from functools import reduce
density_over_time_df = reduce(lambda a, b: a.unionByName(b), all_density_time_dfs)

print("\n✓ Density over time computed (Spark-side, all segments)")
print("="*60)


DENSITY OVER TIME COMPUTATION
Using bounded density metric set: ['tensor_util', 'gpu_util', 'tensor_tflops', 'tensor_tflops_sm', 'chip_power', 'redfish_power', 'sm_active']

Processing density over time for segment dimension: all
  Processing segment value: all

Processing density over time for segment dimension: gen
  Found 2 values for gen
  Processing segment value: current_gen
  Processing segment value: prior_gen

Processing density over time for segment dimension: region_rollup
  Found 2 values for region_rollup
  Processing segment value: NAM
  Processing segment value: EU

Processing density over time for segment dimension: customer_segment
  Found 4 values for customer_segment
  Processing segment value: AI Lab
  Processing segment value: Internal
  Processing segment value: Bigbird
  Processing segment value: External-Other

Processing density over time for segment dimension: product_segment
  Found 4 values for product_segment
  Processing segment value: PCIE
  Processing se

In [16]:
# ========================================
# PERSIST DENSITY OUTPUTS
# ========================================

print("="*60)
print("PERSISTING DENSITY OUTPUTS")
print("="*60)

static_density_table = f"{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_density_bins"
density_over_time_table = f"{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_density_over_time"
bin_metadata_table = f"{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_bin_metadata"

print("[1/4] Writing static density bins...")
print(f"Writing to: {static_density_table}")
static_density_df.write.mode("overwrite").saveAsTable(static_density_table)

print("[2/4] Writing density over time...")
print(f"Writing to: {density_over_time_table}")
density_over_time_df.write.mode("overwrite").saveAsTable(density_over_time_table)

print("[3/4] Writing bin metadata...")
print(f"Writing to: {bin_metadata_table}")
static_bin_metadata_df.write.mode("overwrite").saveAsTable(bin_metadata_table)

print("[4/4] Cleaning up cached density DataFrames...")
for df_name in ["_df_binned", "_df_binned_time"]:
    if df_name in globals():
        try:
            globals()[df_name].unpersist()
            print(f"✓ Unpersisted {df_name}")
        except Exception as e:
            print(f"Note: could not unpersist {df_name}: {e}")

print("" + "="*60)
print("✓ DENSITY OUTPUTS PERSISTED")
print("="*60)
print("Created tables:")
print(f"  1. {static_density_table}")
print(f"  2. {density_over_time_table}")
print(f"  3. {bin_metadata_table}")
print("="*60)


PERSISTING DENSITY OUTPUTS
[1/4] Writing static density bins...
Writing to: sandbox.sandbox_finance.dcgm_eda_density_bins
[2/4] Writing density over time...
Writing to: sandbox.sandbox_finance.dcgm_eda_density_over_time
[3/4] Writing bin metadata...
Writing to: sandbox.sandbox_finance.dcgm_eda_bin_metadata
[4/4] Cleaning up cached density DataFrames...
✓ Unpersisted _df_binned
✓ Unpersisted _df_binned_time
✓ DENSITY OUTPUTS PERSISTED
Created tables:
  1. sandbox.sandbox_finance.dcgm_eda_density_bins
  2. sandbox.sandbox_finance.dcgm_eda_density_over_time
  3. sandbox.sandbox_finance.dcgm_eda_bin_metadata


In [17]:
# ========================================
# TREND SUMMARY FROM SUMMARY TABLE (WITH SEGMENTATION)
# ========================================

summary_time_col_resolved = SUMMARY_TIME_COL
trend_rows = []

segment_groups = get_segment_groups()

for seg_dim, seg_vals in segment_groups:
    print(f"\nProcessing trend summary for segment dimension: {seg_dim}")

    if seg_dim == "all":
        seg_values_to_process = ["all"]
    else:
        distinct_vals = df_summary.select(seg_dim).distinct().collect()
        seg_values_to_process = [row[seg_dim] if row[seg_dim] is not None else "unknown" for row in distinct_vals]
        print(f"  Found {len(seg_values_to_process)} values for {seg_dim}")

    for seg_val in seg_values_to_process:
        # Filter data if segmented
        if seg_dim == "all":
            df_seg = df_summary
        else:
            if seg_val == "unknown":
                df_seg = df_summary.filter(F.col(seg_dim).isNull())
            else:
                df_seg = df_summary.filter(F.col(seg_dim) == seg_val)

        # Compute trends for this segment
        for pct, cols in SUMMARY_PERCENTILE_GROUPS.items():
            if not cols:
                continue
            pdf = df_seg.select(*([summary_time_col_resolved] + cols)).toPandas()
            pdf[summary_time_col_resolved] = pd.to_datetime(pdf[summary_time_col_resolved], errors='coerce')
            pdf = pdf.dropna(subset=[summary_time_col_resolved])
            if pdf.empty:
                continue
            pdf['time_bucket'] = pdf[summary_time_col_resolved].dt.floor('D')
            melted = pdf.melt(id_vars=['time_bucket'], value_vars=cols, var_name='metric_name', value_name='metric_value')
            melted['base_metric'] = melted['metric_name'].map(normalize_metric_name)
            melted['summary_feature_group'] = pct
            melted['segment_dim'] = seg_dim
            melted['segment_value'] = str(seg_val)
            agg = melted.groupby(['time_bucket', 'metric_name', 'base_metric', 'summary_feature_group', 'segment_dim', 'segment_value'], as_index=False).agg(
                observation_count=('metric_value', 'count'),
                mean_value=('metric_value', 'mean'),
                median_value=('metric_value', 'median')
            )
            trend_rows.append(agg)

trend_summary_pdf = pd.concat(trend_rows, ignore_index=True) if trend_rows else pd.DataFrame(columns=[
    'time_bucket', 'metric_name', 'base_metric', 'summary_feature_group',
    'observation_count', 'mean_value', 'median_value', 'segment_dim', 'segment_value'
])
trend_summary_pdf['run_id'] = RUN_ID

trend_summary_schema = T.StructType([
    T.StructField('time_bucket', T.TimestampType(), True),
    T.StructField('metric_name', T.StringType(), False),
    T.StructField('base_metric', T.StringType(), True),
    T.StructField('summary_feature_group', T.StringType(), True),
    T.StructField('observation_count', T.LongType(), True),
    T.StructField('mean_value', T.DoubleType(), True),
    T.StructField('median_value', T.DoubleType(), True),
    T.StructField('run_id', T.StringType(), False),
    T.StructField('segment_dim', T.StringType(), False),
    T.StructField('segment_value', T.StringType(), False),
])

if trend_summary_pdf.empty:
    trend_summary_df = spark.createDataFrame([], schema=trend_summary_schema)
else:
    tmp = trend_summary_pdf.copy()
    tmp['time_bucket'] = pd.to_datetime(tmp['time_bucket'], errors='coerce')
    tmp['observation_count'] = pd.to_numeric(tmp['observation_count'], errors='coerce').fillna(0).astype('int64')
    tmp['mean_value'] = pd.to_numeric(tmp['mean_value'], errors='coerce')
    tmp['median_value'] = pd.to_numeric(tmp['median_value'], errors='coerce')
    for col in ['metric_name', 'base_metric', 'summary_feature_group', 'run_id', 'segment_dim', 'segment_value']:
        tmp[col] = tmp[col].astype('string')

    # Convert to records and scrub timestamps
    records = tmp.to_dict('records')
    for record in records:
        for col in ['time_bucket']:
            if col in record:
                if pd.isna(record[col]):
                    record[col] = None
                elif isinstance(record[col], pd.Timestamp):
                    record[col] = record[col].to_pydatetime()

    trend_summary_df = spark.createDataFrame(records, schema=trend_summary_schema)

print(f"Generated {len(trend_summary_pdf)} trend rows.")



Processing trend summary for segment dimension: all

Processing trend summary for segment dimension: gen
  Found 2 values for gen

Processing trend summary for segment dimension: region_rollup
  Found 2 values for region_rollup

Processing trend summary for segment dimension: customer_segment
  Found 4 values for customer_segment

Processing trend summary for segment dimension: product_segment
  Found 4 values for product_segment

Processing trend summary for segment dimension: is_training
  Found 2 values for is_training
Generated 303255 trend rows.


In [18]:
# ========================================
# PERSIST TREND OUTPUTS
# ========================================

print("="*60)
print("PERSISTING TREND OUTPUTS")
print("="*60)

trend_summary_table = f"{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_trend_summary"
print(f"Writing to: {trend_summary_table}")
trend_summary_df.write.mode("overwrite").saveAsTable(trend_summary_table)
# print(f"✓ Wrote {trend_summary_df.count()} rows")

print("\n" + "="*60)
print("✓ TREND OUTPUTS PERSISTED")
print("="*60)
print(f"\nCreated table:")
print(f"  {trend_summary_table}")
print("="*60)


PERSISTING TREND OUTPUTS
Writing to: sandbox.sandbox_finance.dcgm_eda_trend_summary

✓ TREND OUTPUTS PERSISTED

Created table:
  sandbox.sandbox_finance.dcgm_eda_trend_summary


In [19]:
# ========================================
# PERSISTENCE VALIDATION
# ========================================
# Validate each persisted table by reading it back

print("="*60)
print("PERSISTENCE VALIDATION")
print("="*60)

output_tables = [
    f"{OUTPUT_PREFIX}_raw_sample",
    f"{OUTPUT_PREFIX}_corr_summary",
    f"{OUTPUT_PREFIX}_corr_over_time",
    f"{OUTPUT_PREFIX}_density_bins",
    f"{OUTPUT_PREFIX}_density_over_time",
    f"{OUTPUT_PREFIX}_bin_metadata",
    f"{OUTPUT_PREFIX}_trend_summary"
]
required_key_columns = {
    f"{OUTPUT_PREFIX}_raw_sample": [RAW_TIME_COL] + CORE_RAW_METRIC_COLS[:2] + ["segment_dim", "segment_value"],
    f"{OUTPUT_PREFIX}_corr_summary": ["metric_x", "metric_y", "method", "summary_feature_group", "segment_dim", "segment_value"],
    f"{OUTPUT_PREFIX}_corr_over_time": ["time_bucket", "metric_x", "metric_y", "method", "segment_dim", "segment_value"],
    f"{OUTPUT_PREFIX}_density_bins": ["metric_x", "metric_y", "x_bin", "y_bin", "count", "segment_dim", "segment_value"],
    f"{OUTPUT_PREFIX}_density_over_time": ["time_bucket", "metric_x", "metric_y", "x_bin", "y_bin", "segment_dim", "segment_value"],
    f"{OUTPUT_PREFIX}_bin_metadata": ["metric_name", "bin_number", "bin_min", "bin_max", "segment_dim", "segment_value"],
    f"{OUTPUT_PREFIX}_trend_summary": ["time_bucket", "metric_name", "summary_feature_group", "segment_dim", "segment_value"]
}

print(f"\nValidating {len(output_tables)} persisted tables...\n")
validation_results = []
for table_name in output_tables:
    full_table_name = f"{OUTPUT_SCHEMA}.{table_name}"
    print(f"[{table_name}]")
    try:
        df_check = spark.table(full_table_name)
        schema_str = ", ".join([f"{field.name}:{field.dataType.simpleString()}" for field in df_check.schema.fields[:10]])
        if len(df_check.schema.fields) > 10:
            schema_str += f", ... ({len(df_check.schema.fields)} total)"
        missing_keys = [c for c in required_key_columns.get(table_name, []) if c not in df_check.columns]
        print(f"  ✓ Schema: {schema_str}")
        print("  ✓ Table readable")
        if missing_keys:
            print(f"  ✗ Missing expected key columns: {missing_keys}")
            status = "FAIL"
        else:
            print("  ✓ Expected key columns present")
            status = "PASS"
        validation_results.append({"table": table_name, "status": status})
    except Exception as e:
        print(f"  ✗ ERROR: {e}")
        validation_results.append({"table": table_name, "status": "FAIL"})
    print()

print("="*60)
print("VALIDATION SUMMARY")
print("="*60)
for result in validation_results:
    status_symbol = "✓" if result["status"] == "PASS" else "✗"
    print(f"{status_symbol} {result['table']}: {result['status']}")
all_passed = all(r["status"] == "PASS" for r in validation_results)
print("\n" + "="*60)
if all_passed:
    print("✓ ALL TABLES VALIDATED SUCCESSFULLY")
else:
    print("✗ SOME TABLES FAILED VALIDATION - REVIEW ABOVE")
print("="*60)


PERSISTENCE VALIDATION

Validating 7 persisted tables...

[dcgm_eda_raw_sample]
  ✓ Schema: datestamp:timestamp, tensor_util:double, gpu_util:double, tensor_tflops:double, tensor_tflops_sm:double, chip_power:double, redfish_power:double, sm_active:double, dram_active:double, mem_copy_util:double, ... (29 total)
  ✓ Table readable
  ✓ Expected key columns present

[dcgm_eda_corr_summary]
  ✓ Schema: metric_x:string, metric_y:string, base_metric_x:string, base_metric_y:string, method:string, summary_percentile:string, summary_feature_group:string, correlation_value:double, obs_count:bigint, run_id:string, ... (14 total)
  ✓ Table readable
  ✓ Expected key columns present

[dcgm_eda_corr_over_time]
  ✓ Schema: time_bucket:timestamp, metric_x:string, metric_y:string, base_metric_x:string, base_metric_y:string, correlation_value:double, method:string, summary_percentile:string, run_id:string, time_grain:string, ... (12 total)
  ✓ Table readable
  ✓ Expected key columns present

[dcgm_eda_de

In [20]:
# ========================================
# COMPREHENSIVE READ-BACK VALIDATION
# ========================================
# Final validation: Demonstrate downstream notebook can consume outputs

print("="*60)
print("COMPREHENSIVE READ-BACK VALIDATION")
print("="*60)
print("Simulating downstream notebook reading persisted outputs...")
print("All checks remain scan-safe: no full-table counts on large outputs.")

print("\n[Example Downstream Workflow]")

print("\n1. Reading correlation summary...")
corr_summary = spark.table(f"{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_corr_summary")
print("   Correlation summary table readable")
print("   Includes summary_feature_group and base_metric mapping metadata")

print("\n2. Reading correlation over time...")
corr_over_time = spark.table(f"{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_corr_over_time")
if len(SUMMARY_METRIC_COLS) >= 2:
    _ = corr_over_time.filter((F.col("metric_x") == SUMMARY_METRIC_COLS[0]) & (F.col("metric_y") == SUMMARY_METRIC_COLS[1])).orderBy("time_bucket")
print("   Example correlation-over-time query constructed successfully")

print("\n3. Reading bin metadata...")
bin_metadata = spark.table(f"{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_bin_metadata")
print("   Bin metadata readable for density reconstruction")

print("\n4. Reading static density bins...")
density_bins = spark.table(f"{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_density_bins")
print("   Static density bins readable")

print("\n5. Reading density over time...")
density_over_time = spark.table(f"{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_density_over_time")
print("   Density-over-time table readable with fixed-bin design")

print("\n6. Reading trend summary...")
trend_summary = spark.table(f"{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_trend_summary")
print("   Trend summary readable with summary time/feature metadata")

print("\n" + "="*60)
print("✓ COMPREHENSIVE VALIDATION COMPLETE")
print("="*60)
print("\nConclusion:")
print("  ✓ All persisted tables are readable")
print("  ✓ Schemas are queryable")
print("  ✓ Downstream notebooks can consume these outputs")
print("  ✓ Validation remains intentionally lightweight to avoid full scans")
print("="*60)


# ========================================
# SEGMENT FILTERING EXAMPLES
# ========================================

print("\n" + "="*60)
print("SEGMENT FILTERING EXAMPLES")
print("="*60)

print("\n[Example 1] Get unsegmented baseline data")
print("Filter: WHERE segment_dim = 'all' AND segment_value = 'all'")
baseline_corr = spark.table(f"{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_corr_summary").filter(
    (F.col("segment_dim") == "all") & (F.col("segment_value") == "all")
)
print(f"Baseline correlation rows: {baseline_corr.count()}")

print("\n[Example 2] Get gen-segmented data")
print("Filter: WHERE segment_dim = 'gen'")
gen_segments = spark.table(f"{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_corr_summary").filter(
    F.col("segment_dim") == "gen"
)
distinct_gen_values = gen_segments.select("segment_value").distinct().collect()
print(f"Gen segment values: {[row.segment_value for row in distinct_gen_values]}")

print("\n[Example 3] Get specific segment value")
print("Filter: WHERE segment_dim = 'region_rollup' AND segment_value = '<specific_region>'")
print("(This would return data for a specific region)")

print("\n" + "="*60)
print("✓ SEGMENT FILTERING EXAMPLES COMPLETE")
print("="*60)


COMPREHENSIVE READ-BACK VALIDATION
Simulating downstream notebook reading persisted outputs...
All checks remain scan-safe: no full-table counts on large outputs.

[Example Downstream Workflow]

1. Reading correlation summary...
   Correlation summary table readable
   Includes summary_feature_group and base_metric mapping metadata

2. Reading correlation over time...
   Example correlation-over-time query constructed successfully

3. Reading bin metadata...
   Bin metadata readable for density reconstruction

4. Reading static density bins...
   Static density bins readable

5. Reading density over time...
   Density-over-time table readable with fixed-bin design

6. Reading trend summary...
   Trend summary readable with summary time/feature metadata

✓ COMPREHENSIVE VALIDATION COMPLETE

Conclusion:
  ✓ All persisted tables are readable
  ✓ Schemas are queryable
  ✓ Downstream notebooks can consume these outputs
  ✓ Validation remains intentionally lightweight to avoid full scans

SE

In [21]:
# ========================================
# SPARK CLUSTER DIAGNOSTICS
# ========================================

from pyspark import SparkContext

print("="*60)
print("SPARK CLUSTER DIAGNOSTICS")
print("="*60)

sc = spark.sparkContext

print("\n1. CONFIGURATION:")
print(f"   App Name: {sc.appName}")
print(f"   Spark Version: {sc.version}")
print(f"   Master: {sc.master}")
print(f"   Default Parallelism: {sc.defaultParallelism}")

print("\n2. EXECUTOR INFO:")
try:
    executor_memory_map = dict(sc._jsc.sc().getExecutorMemoryStatus())
    num_executors = len(executor_memory_map) - 1
    print(f"   Number of Executors: {num_executors}")
except Exception as e:
    print(f"   Could not get executor count: {e}")

conf = sc.getConf()
print(f"   Driver Memory: {conf.get('spark.driver.memory', 'Not set')}")
print(f"   Executor Memory: {conf.get('spark.executor.memory', 'Not set')}")
print(f"   Executor Cores: {conf.get('spark.executor.cores', 'Not set')}")

print("\n3. STORAGE INFO:")
try:
    storage_status = sc._jsc.sc().getRDDStorageInfo()
    print(f"   Number of cached RDDs: {storage_status.length()}")
except Exception as e:
    print(f"   Could not get storage info: {e}")

print("\n4. SPARK UI:")
try:
    ui_url = sc.uiWebUrl
    if ui_url:
        print(f"   Spark UI URL: {ui_url}")
except Exception:
    print(f"   Spark UI not available")

print("\n" + "="*60)


SPARK CLUSTER DIAGNOSTICS

1. CONFIGURATION:
   App Name: sparkcaster-jbok-4df513e2-3d70-43e2-b455-0630944c9842
   Spark Version: 4.1.2
   Master: k8s://https://10.16.0.1:443
   Default Parallelism: 40

2. EXECUTOR INFO:
   Could not get executor count: JavaObject.keys() returned a non-iterable (type JavaObject)
   Driver Memory: 16g
   Executor Memory: 64g
   Executor Cores: 4

3. STORAGE INFO:
   Could not get storage info: An error occurred while calling o45465.length. Trace:
py4j.Py4JException: Method length([]) does not exist
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:321)
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:329)
	at py4j.Gateway.invoke(Gateway.java:274)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.execute(CallCommand.java:79)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184)
	at py4j.ClientServerConnection.run(ClientServerConnection.j

In [22]:
# ========================================
# FINAL CLEANUP
# ========================================

print("="*60)
print("FINAL CLEANUP")
print("="*60)

for df_name in ["df_raw_sample", "df_summary_clean", "_df_binned", "_df_binned_time", "_density_base"]:
    if df_name in globals():
        try:
            globals()[df_name].unpersist()
            print(f"✓ Unpersisted {df_name}")
        except Exception as e:
            print(f"Note: could not unpersist {df_name}: {e}")

try:
    spark.catalog.clearCache()
    print("✓ Cleared Spark cache")
except Exception as e:
    print(f"Note: could not clear Spark cache: {e}")

print("="*60)
print("✓ CLEANUP COMPLETE")
print("="*60)


FINAL CLEANUP
✓ Unpersisted df_raw_sample
✓ Unpersisted df_summary_clean
✓ Unpersisted _df_binned
✓ Unpersisted _df_binned_time
✓ Unpersisted _density_base
✓ Cleared Spark cache
✓ CLEANUP COMPLETE


---

## Corr_Summary Segmentation Fix - Documentation

### Root Cause

The `build_all_summary_corrs_segmented()` function was attempting to discover and filter segment values from `df_summary_clean` using operations like:
- `df_summary_clean.select(seg_dim).distinct().collect()`
- `df_summary_clean.filter(F.col(seg_dim) == seg_val)`

However, `df_summary_clean` was built by selecting only `summary_keep_cols`, which originally contained:
- Time column (`SUMMARY_TIME_COL`)
- Summary metric columns (`SUMMARY_ALL_FEATURE_COLS`)

The segment dimension columns from `SEGMENT_DIMS` were **not included** in `summary_keep_cols`, causing the segmented correlation logic to fail when trying to access columns that didn't exist in the DataFrame.

### What Was Changed

**Location:** Cell #9 (SUMMARY CORRELATIONS: P50 / P95 / P99)

**Changes made:**

1. **Segment Column Discovery** (after `summary_keep_cols` construction):
   - Added code to discover which `SEGMENT_DIMS` columns exist in `df_summary`
   - Appended these columns to `summary_keep_cols` before creating `df_summary_clean`
   - Added informative print statements showing which segment columns were added

2. **Pre-Build Validation** (after `df_summary_clean` creation):
   - Added validation showing which segment columns are available in `df_summary_clean`
   - Added bounded sample collection showing example values for first 2 segment dimensions
   - Helps verify the fix worked before expensive correlation computation

3. **Post-Build Validation** (after `corr_summary_pdf` creation):
   - Added comprehensive validation of the output structure
   - Validates `segment_dim` and `segment_value` columns exist
   - Shows row counts for baseline (all/all) and each segmented dimension
   - Displays sample segment values to prove segmentation worked

### Fix Method

**Expanded `summary_keep_cols` to include segment columns.**

This approach was chosen because:
- Minimal change to existing architecture
- Preserves the single `df_summary_clean` DataFrame contract
- No need for separate DataFrames or complex branching logic
- Maintains backward compatibility (empty `SEGMENT_DIMS` = no extra columns)

### What Downstream Notebooks Can Now Assume

**The `dcgm_eda_corr_summary` table now reliably contains:**

1. **Baseline rows** with `segment_dim = 'all'` and `segment_value = 'all'`
   - Global correlations across all data (preserves original behavior)

2. **Segmented rows** for each dimension in `SEGMENT_DIMS`:
   - `segment_dim = 'gen'` with values for each GPU generation
   - `segment_dim = 'region_rollup'` with values 'EU' and 'NAM'
   - `segment_dim = 'customer_segment'` with customer segment values
   - `segment_dim = 'product_segment'` with product segment values
   - `segment_dim = 'is_training'` with training status values

3. **Segmentation is one-dimensional** (not cross-product):
   - Each row belongs to exactly one segment dimension
   - No multi-dimensional combinations (e.g., no gen × region × customer)

4. **All original correlation columns preserved**:
   - `metric_x`, `metric_y`
   - `correlation_value`, `method`
   - `summary_percentile`
   - Plus the new `segment_dim` and `segment_value`

**Downstream filtering examples:**

```python
# Get baseline global correlations
baseline = df_corr.filter((F.col("segment_dim") == "all") & (F.col("segment_value") == "all"))

# Get all gen-segmented correlations
gen_corrs = df_corr.filter(F.col("segment_dim") == "gen")

# Get correlations for a specific region
eu_corrs = df_corr.filter((F.col("segment_dim") == "region_rollup") & (F.col("segment_value") == "EU"))
```

### Testing Status

✅ **Validation code added** to verify the fix works at runtime  
✅ **No changes to segmentation logic** - only fixed the input DataFrame contract  
✅ **Backward compatible** - empty `SEGMENT_DIMS` still works (produces baseline only)  

---